# Phase 2B -- Voice Engine depth pass: honest accuracy, not shortcut accuracy

Implements `docs/PHASE_2B_VOICE_DEPTH.md`. Generated from `notebooks/src/04_voice_depth.py` by
`scripts/build_notebooks.py` -- edit the source, not this .ipynb.

- **Task 1 -- more data:** every bonafide clip in the LA evaluation partition (~7.3k, vs 619 in Phase 2)
  plus a capped sample of spoof clips, decoded once into an on-disk feature cache.
- **Task 2 -- attack-disjoint split:** attack IDs come from the official protocol file shipped in the
  same HuggingFace repo, and are used only after it reproduces every clip's label and speaker exactly.
  Shared-attack and attack-disjoint models are scored on *the same test clips*.
- **Task 3 -- one variable per run:** A = LCNN on the larger data; B = A + augmentation; C = frozen
  XLS-R front-end + layer-weighted head. Selection on validation only; test touched once per model.
- **Task 4 -- silence:** trimmed stays the headline; untrimmed is labelled a shortcut. Four ablations
  separate silence *duration* from silence *content*.
- **Task 5 -- rigour:** 3 seeds for the headline and best model, stratified bootstrap 95% CIs (1000
  resamples) on every test EER, class counts next to every number.
- **Task 6 -- In-the-Wild:** rerun with the fixed loader, >= 100 clips per class, failures recorded.

Nothing here is compared with published ASVspoof numbers: this is not the official protocol.

**Kaggle:** GPU (T4), Internet on. Recommended: skip the interactive Run All and go straight to
**Save Version -> Save & Run All (Commit)** -- a commit re-runs everything from scratch anyway, so an
interactive run first doubles the GPU time. See `docs/kaggle_setup.md`.

## Setup

In [ ]:
# Kaggle: Accelerator = GPU, Internet = on.
import os
import subprocess
import sys

# True only when tests/smoke_notebook_04.py runs this notebook's code on synthetic data.
SMOKE_TEST = os.environ.get("TRUSTMESH_SMOKE") == "1"


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


if not SMOKE_TEST:
    for module_name, package in [("datasets", "datasets"), ("soundfile", "soundfile"),
                                 ("transformers", "transformers"), ("fsspec", "fsspec")]:
        try:
            __import__(module_name)
        except ImportError:
            pip_install(package)

In [ ]:
import json
import math
import shutil
import time
import traceback
from pathlib import Path

import numpy as np
import torch

RUN_START = time.time()
SEED = 42
SEEDS = [42, 43, 44]
torch.manual_seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path(os.environ.get("TRUSTMESH_OUTPUT_DIR", "./artifacts/voice"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# The spec asks for the cache in /kaggle/working so a crash doesn't force re-decoding. It is deleted at
# the end of a *successful* run (DELETE_CACHE_AT_END) so the committed output stays small.
CACHE_DIR = Path(os.environ.get("TRUSTMESH_CACHE_DIR", str(OUTPUT_DIR / "cache")))
DELETE_CACHE_AT_END = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() and not SMOKE_TEST else "cpu")

SPOOF_PER_BONAFIDE = 4           # all bonafide + 4 spoof per bonafide (spec: 1:4 to 1:6). Lower this, never cut bonafide.
CLIP_SECONDS = 4.0
SILENCE_WINDOW_S = 0.2           # silence-only ablation: exactly this much genuine silence per clip (21 LFCC frames)
FIXED_SILENCE_S = 0.5            # zeros_fixed / noise_fixed ablations
NOISE_DBFS = -60.0
TEST_FRACTION, VAL_FRACTION = 0.3, 0.2
N_HELD_OUT_ATTACKS = 4           # of the 13 LA eval attacks (A07-A19)

EPOCHS = 3 if SMOKE_TEST else 25
PATIENCE = 5
BATCH_SIZE = 8 if SMOKE_TEST else 64
LEARNING_RATE = 3e-4
SSL_MODEL = "facebook/wav2vec2-xls-r-300m"
SSL_FALLBACK_MODEL = "facebook/wav2vec2-base"
SSL_BATCH = 4 if SMOKE_TEST else 16
SSL_EPOCHS = 3 if SMOKE_TEST else 60
SSL_LR, SSL_PATIENCE = 1e-3, 8
# Phase 2C Model F re-runs the frozen encoder every training batch (on-the-fly augmentation), so one
# epoch costs far more wall-clock than Model C's cached-feature training -- Model C historically
# converged within 13-44 epochs (Phase 2B), so this caps Model F's worst case without changing its
# early-stopping patience. Same protocol otherwise (lr, patience, batch size).
SSL_EPOCHS_ONTHEFLY = 3 if SMOKE_TEST else 30
# Model F is Phase 2C's most expensive run by a wide margin (it re-runs the frozen encoder on every
# training batch, even after issues #26/#27's fixes) and is a smaller, more speculative check than D or
# E. Off by default to keep a real Kaggle attempt to a couple of hours (user's call, 2026-09-30, after
# three slow/stuck attempts); set TRUSTMESH_RUN_MODEL_F=1 to include it. The smoke test always sets this
# so Model F's code path stays covered regardless of this default.
RUN_MODEL_F = SMOKE_TEST or os.environ.get("TRUSTMESH_RUN_MODEL_F") == "1"
# Model E's full-dataset sequence cache (~15 GB) overran this Kaggle environment's real disk quota even
# after Task 4's cleanup freed 7.17 GB -- a second, distinct disk failure (issue #28) mid-way through
# building it. Off by default for the same reason as Model F, until the disk budget is worked out
# (e.g. a smaller clip subset just for this model); set TRUSTMESH_RUN_MODEL_E=1 to include it.
RUN_MODEL_E = SMOKE_TEST or os.environ.get("TRUSTMESH_RUN_MODEL_E") == "1"

N_ITW_PER_CLASS = 3 if SMOKE_TEST else 100
N_BOOTSTRAP = 50 if SMOKE_TEST else 1000
PHASE2_TEST_COUNTS = {"bonafide": 182, "spoof": 1643}   # Phase 2's headline test set, for the CI-shrinkage check
TIME_BUDGET_HOURS = 10.0                                # Kaggle sessions stop at 12 h

CLIP_SAMPLES = int(16000 * CLIP_SECONDS)
SILENCE_SAMPLES = int(16000 * SILENCE_WINDOW_S)
RESULTS_PATH = OUTPUT_DIR / "results_phase2b.json"
RESULTS = {"smoke_test": SMOKE_TEST, "not_run": {}}
# Phase 2C (docs/PHASE_2C_VOICE_SSL_DEPTH.md) writes its own file, never merged into results_phase2b.json,
# so the two phases stay separable per the spec.
RESULTS2C_PATH = OUTPUT_DIR / "results_phase2c.json"
RESULTS2C = {"smoke_test": SMOKE_TEST, "not_run": {}}


def save_progress(key, value):
    RESULTS[key] = value
    RESULTS_PATH.write_text(json.dumps(RESULTS, indent=2, default=float), encoding="utf-8")


def save_progress_2c(key, value):
    RESULTS2C[key] = value
    RESULTS2C_PATH.write_text(json.dumps(RESULTS2C, indent=2, default=float), encoding="utf-8")


def elapsed_hours():
    return (time.time() - RUN_START) / 3600


def within_budget(step):
    """Optional steps are skipped (and recorded as not run) rather than risking the whole session."""
    if elapsed_hours() < TIME_BUDGET_HOURS:
        return True
    RESULTS["not_run"][step] = f"time budget: {elapsed_hours():.1f} h elapsed of {TIME_BUDGET_HOURS} h"
    save_progress("not_run", RESULTS["not_run"])
    print(f"SKIPPED {step}: {RESULTS['not_run'][step]}")
    return False


def within_budget_2c(step):
    """Same budget check as `within_budget`, for Phase 2C steps -- records into results_phase2c.json's
    own `not_run`, not Phase 2B's, since Phase 2C runs in the same Kaggle session against the same
    TIME_BUDGET_HOURS clock but must not write into results_phase2b.json."""
    if elapsed_hours() < TIME_BUDGET_HOURS:
        return True
    RESULTS2C["not_run"][step] = f"time budget: {elapsed_hours():.1f} h elapsed of {TIME_BUDGET_HOURS} h"
    save_progress_2c("not_run", RESULTS2C["not_run"])
    print(f"SKIPPED {step}: {RESULTS2C['not_run'][step]}")
    return False


print("device:", DEVICE, "| SMOKE_TEST:", SMOKE_TEST, "| torch", torch.__version__)

## Shared voice-engine code (inlined from `engines/voice/`)

In [ ]:
# ---- inlined from engines/voice/data.py (edit the module, then rebuild) ----
"""ASVspoof 2019 LA loading and train/test splits.

Dataset: `SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA` on HuggingFace -- parquet, no loading script,
71,237 clips, 16 kHz FLAC. IMPORTANT: it contains only the **evaluation** partition of LA
(7,355 bonafide / 63,882 spoof). The HF copy carrying the official train/dev/eval partitions
(`LanceaKing/asvspoof2019`) needs `trust_remote_code`, which current `datasets` versions no longer
support, so the official protocol (train on attacks A01-A06, test on unseen A07-A19) is not
reachable here. We therefore split the eval partition ourselves; see `speaker_disjoint_split` for
why the split is speaker-disjoint and what that does and does not fix.

Clips are streamed (not downloaded in full) so the subset size is a knob, and audio is fetched
undecoded and decoded here with soundfile -- decoding through the `datasets` Audio feature changed
backends across versions, and doing it ourselves keeps this working either way.
"""
import io
import json

import numpy as np

ASVSPOOF_DATASET = "SpeechAntiSpoofingBenchmarks/ASVspoof2019_LA"
ASVSPOOF_SPLIT = "test"  # the dataset's only split: the LA evaluation partition
IN_THE_WILD_DATASET = "mueller91/In-The-Wild"
TARGET_SR = 16000
BONAFIDE, SPOOF = 1, 0  # label 1 = bonafide = the positive/target class for EER


def _decode_audio(blob):
    """Decode an undecoded `datasets` audio value to (float32 mono waveform, sample rate)."""
    import soundfile as sf

    if isinstance(blob, dict) and blob.get("bytes"):
        wave, sr = sf.read(io.BytesIO(blob["bytes"]), dtype="float32", always_2d=False)
    elif isinstance(blob, dict) and blob.get("array") is not None:
        wave, sr = np.asarray(blob["array"], dtype=np.float32), int(blob["sampling_rate"])
    else:
        path = blob["path"] if isinstance(blob, dict) else blob
        if isinstance(path, str) and "://" in path:
            # Archive-backed datasets (e.g. In-the-Wild, shipped as one zip) come back as chained fsspec
            # URLs like "zip://a.wav::hf://datasets/.../x.zip", which libsndfile can't open. fsspec can,
            # and it only range-reads the needed zip member, not the whole archive.
            import fsspec

            with fsspec.open(path, "rb") as handle:
                wave, sr = sf.read(io.BytesIO(handle.read()), dtype="float32", always_2d=False)
        else:
            wave, sr = sf.read(path, dtype="float32", always_2d=False)
    if wave.ndim > 1:
        wave = wave.mean(axis=1)
    return np.ascontiguousarray(wave, dtype=np.float32), int(sr)


def load_asvspoof_subset(n_clips, seed=42, shuffle_buffer=20000, dataset=ASVSPOOF_DATASET,
                         split=ASVSPOOF_SPLIT, progress_every=500):
    """Stream a shuffled subset of ASVspoof 2019 LA. Returns a list of clip dicts."""
    from datasets import Audio, load_dataset

    stream = load_dataset(dataset, split=split, streaming=True)
    stream = stream.cast_column("audio", Audio(decode=False))
    stream = stream.shuffle(seed=seed, buffer_size=shuffle_buffer).take(n_clips)

    clips = []
    for row in stream:
        wave, sr = _decode_audio(row["audio"])
        try:
            notes = json.loads(row.get("notes") or "{}")
        except (json.JSONDecodeError, TypeError):
            notes = {}
        # The dataset's ClassLabel is 0=bonafide, 1=spoof; we flip so 1 = bonafide = target class.
        clips.append({
            "waveform": wave,
            "sample_rate": sr,
            "label": BONAFIDE if int(row["label"]) == 0 else SPOOF,
            "speaker_id": notes.get("speaker_id") or "unknown",
            "utterance_id": notes.get("utterance_id") or row.get("path") or f"clip_{len(clips)}",
            "path": row.get("path"),
        })
        if progress_every and len(clips) % progress_every == 0:
            print(f"  loaded {len(clips)}/{n_clips} clips")
    return clips


def load_in_the_wild_clips(n_clips, seed=42, shuffle_buffer=2000, dataset=IN_THE_WILD_DATASET):
    """A handful of out-of-domain clips for the generalization spot-check. Column names on this
    dataset aren't documented, so the audio/label columns are detected rather than assumed."""
    from datasets import Audio, load_dataset

    stream = load_dataset(dataset, split="train", streaming=True)
    columns = list(next(iter(stream.take(1))).keys())
    audio_col = next((c for c in columns if "audio" in c.lower() or c.lower() in ("file", "path")), None)
    label_col = next((c for c in columns if c.lower() in ("label", "key", "class", "type", "target")), None)
    if audio_col is None:
        raise RuntimeError(f"no audio-looking column in {columns}")

    try:
        stream = stream.cast_column(audio_col, Audio(decode=False))
    except (ValueError, TypeError):
        pass
    stream = stream.shuffle(seed=seed, buffer_size=shuffle_buffer).take(n_clips)

    clips = []
    for row in stream:
        wave, sr = _decode_audio(row[audio_col])
        raw_label = row.get(label_col)
        label = None
        if isinstance(raw_label, str):
            label = BONAFIDE if raw_label.strip().lower() in ("bona-fide", "bonafide", "real", "genuine") else SPOOF
        elif isinstance(raw_label, (int, np.integer)):
            label = int(raw_label)  # unknown polarity; recorded but flagged in the results
        clips.append({"waveform": wave, "sample_rate": sr, "label": label,
                      "raw_label": raw_label, "label_column": label_col, "speaker_id": "unknown",
                      "utterance_id": f"itw_{len(clips)}", "path": row.get("path")})
    return clips


ASVSPOOF_PROTOCOL = "protocols/ASVspoof2019.LA.cm.eval.trl.txt"
IN_THE_WILD_ARCHIVE = "datasets/mueller91/In-The-Wild/release_in_the_wild.zip"
_BONAFIDE_WORDS = {"bona-fide", "bonafide", "bona fide", "real", "genuine"}
_SPOOF_WORDS = {"spoof", "fake", "deepfake", "synthetic"}


def load_asvspoof_table(dataset=ASVSPOOF_DATASET, split=ASVSPOOF_SPLIT, cache_dir=None):
    """Download the whole parquet once and return it with audio left undecoded.

    Phase 2 streamed a shuffled sample. Phase 2B keeps *every* bonafide clip, which means touching
    every row anyway, and random access by index into a local memory-mapped table beats a streamed
    shuffle buffer for that (streaming, not training, was the slow part of Phase 2).
    """
    from datasets import Audio, load_dataset

    table = load_dataset(dataset, split=split, cache_dir=cache_dir)
    return table.cast_column("audio", Audio(decode=False))


def table_metadata(table):
    """Labels, utterance and speaker IDs for every row, without decoding any audio."""
    labels = np.array([BONAFIDE if int(v) == 0 else SPOOF for v in table["label"]])
    notes = []
    for raw in table["notes"]:
        try:
            notes.append(json.loads(raw or "{}"))
        except (json.JSONDecodeError, TypeError):
            notes.append({})
    paths = list(table["path"])
    utterances = [n.get("utterance_id") or (p.rsplit(".", 1)[0] if p else f"row_{i}")
                  for i, (n, p) in enumerate(zip(notes, paths))]
    return {
        "label": labels,
        "utterance_id": utterances,
        "speaker_id": [n.get("speaker_id") or "unknown" for n in notes],
        "columns": list(table.features),
        "note_keys": sorted({k for n in notes for k in n}),
    }


def decode_row(table, index):
    return _decode_audio(table[int(index)]["audio"])


def load_eval_protocol(path=None, dataset=ASVSPOOF_DATASET, filename=ASVSPOOF_PROTOCOL):
    """Official ASVspoof CM protocol: one line per clip, `SPEAKER UTTERANCE - ATTACK KEY`.

    The parquet has no attack column, but the same HF repo ships this file, which is the source of
    attack IDs (A07-A19). It is only trusted after `verify_protocol` passes.
    """
    if path is None:
        from huggingface_hub import hf_hub_download

        path = hf_hub_download(repo_id=dataset, filename=filename, repo_type="dataset")
    protocol = {}
    with open(path, encoding="utf-8") as handle:
        for line in handle:
            parts = line.split()
            if len(parts) != 5:
                continue
            speaker, utterance, _, attack, key = parts
            protocol[utterance] = {
                "speaker_id": speaker,
                "attack": "bonafide" if attack == "-" else attack,
                "label": BONAFIDE if key == "bonafide" else SPOOF,
            }
    return protocol, str(path)


def verify_protocol(protocol, utterance_ids, labels, speaker_ids):
    """The join is only usable if it reproduces the dataset's own labels and speakers exactly --
    a protocol file for a different partition, or misaligned IDs, fails this loudly."""
    n = len(utterance_ids)
    found = label_ok = speaker_ok = 0
    bonafide_clean = True
    for utterance, label, speaker in zip(utterance_ids, labels, speaker_ids):
        entry = protocol.get(utterance)
        if entry is None:
            continue
        found += 1
        label_ok += entry["label"] == label
        speaker_ok += entry["speaker_id"] == speaker
        if label == BONAFIDE and entry["attack"] != "bonafide":
            bonafide_clean = False
    return {
        "clips": n,
        "found_in_protocol": found,
        "label_agreement": label_ok,
        "speaker_agreement": speaker_ok,
        "bonafide_rows_have_no_attack": bonafide_clean,
        "usable": found == n and label_ok == n and speaker_ok == n and bonafide_clean,
    }


def select_subset(labels, spoof_per_bonafide, seed=42):
    """Every bonafide clip plus a random sample of spoof clips capped at `spoof_per_bonafide` per
    bonafide clip. If a budget forces a smaller set, lower the ratio -- bonafide is never cut."""
    labels = np.asarray(labels)
    bonafide = np.flatnonzero(labels == BONAFIDE)
    spoof = np.flatnonzero(labels == SPOOF)
    n_spoof = min(len(spoof), int(round(spoof_per_bonafide * len(bonafide))))
    rng = np.random.default_rng(seed)
    return np.sort(np.concatenate([bonafide, rng.choice(spoof, n_spoof, replace=False)]))


def choose_held_out_attacks(attacks, n_held_out, seed=42):
    spoof_attacks = sorted({a for a in attacks if a != "bonafide"})
    n = max(1, min(n_held_out, len(spoof_attacks) - 1))  # leave at least one attack for training
    rng = np.random.default_rng(seed)
    return sorted(rng.choice(spoof_attacks, size=n, replace=False).tolist())


def drop_attacks(indices, attacks, held_out):
    """Train/val side of the attack-disjoint protocol: bonafide plus spoofs from non-held-out attacks."""
    held = set(held_out)
    return [i for i in indices if attacks[i] == "bonafide" or attacks[i] not in held]


def keep_only_attacks(indices, attacks, held_out):
    """Test side of the attack-disjoint protocol: bonafide plus spoofs from held-out attacks only."""
    held = set(held_out)
    return [i for i in indices if attacks[i] == "bonafide" or attacks[i] in held]


class ZipAudioArchive:
    """Members of a zip that may be remote (hf://, https://), read without downloading it whole.

    One zip filesystem is opened and reused, so the central directory of a multi-GB archive is read
    once and each clip is a single range read -- reopening a chained `zip://...::hf://...` URL per clip
    (what Phase 2's loader effectively did) re-reads the directory every time.
    """

    def __init__(self, archive, protocol="hf"):
        import fsspec

        self.archive, self.protocol = archive, protocol
        self.fs = fsspec.filesystem("zip", fo=archive, target_protocol=protocol)

    def members(self):
        return sorted(self.fs.find(""))

    def read_bytes(self, member):
        with self.fs.open(member, "rb") as handle:
            return handle.read()


def read_archive_metadata(archive):
    """Find the label metadata inside the archive (In-the-Wild ships `meta.csv` with file/label
    columns) rather than assuming a HuggingFace column schema that isn't documented."""
    import pandas as pd

    members = archive.members()
    csvs = [m for m in members if m.lower().endswith(".csv")]
    if not csvs:
        raise RuntimeError(f"no metadata .csv among {len(members)} archive members")
    meta_member = next((m for m in csvs if m.rsplit("/", 1)[-1].lower() == "meta.csv"), csvs[0])
    df = pd.read_csv(io.BytesIO(archive.read_bytes(meta_member)))

    def looks_like_files(col):
        return df[col].astype(str).str.lower().str.endswith((".wav", ".flac", ".mp3")).mean() > 0.9

    def looks_like_labels(col):
        values = set(df[col].astype(str).str.strip().str.lower().unique())
        return bool(values) and values <= (_BONAFIDE_WORDS | _SPOOF_WORDS)

    file_col = next((c for c in df.columns if looks_like_files(c)), None)
    label_col = next((c for c in df.columns if looks_like_labels(c)), None)
    if file_col is None or label_col is None:
        raise RuntimeError(f"could not identify file/label columns in {meta_member}: {list(df.columns)}")

    prefix = meta_member.rsplit("/", 1)[0] + "/" if "/" in meta_member else ""
    raw = df[label_col].astype(str).str.strip()
    table = pd.DataFrame({
        "member": prefix + df[file_col].astype(str),
        "raw_label": raw,
        "label": np.where(raw.str.lower().isin(_BONAFIDE_WORDS), BONAFIDE, SPOOF),
    })
    info = {"metadata_member": meta_member, "file_column": file_col, "label_column": label_col,
            "raw_label_values": sorted(raw.unique().tolist()), "rows": int(len(table)),
            "rows_bonafide": int((table.label == BONAFIDE).sum()), "rows_spoof": int((table.label == SPOOF).sum())}
    return table, info


def load_archive_clips_balanced(archive, n_per_class, seed=42, max_failures=None):
    """`n_per_class` clips of each class, read one by one; every failure is recorded, not hidden."""
    import soundfile as sf

    table, info = read_archive_metadata(archive)
    max_failures = max_failures if max_failures is not None else 2 * n_per_class
    rng = np.random.default_rng(seed)
    clips, failures = [], []
    for label in (BONAFIDE, SPOOF):
        pool = table[table.label == label].reset_index(drop=True)
        loaded = 0
        for pos in rng.permutation(len(pool)):
            if loaded >= n_per_class:
                break
            row = pool.iloc[int(pos)]
            try:
                wave, sr = sf.read(io.BytesIO(archive.read_bytes(row.member)), dtype="float32", always_2d=False)
            except Exception as exc:  # noqa: BLE001 -- recorded per clip, and capped below
                failures.append({"member": row.member, "error": f"{type(exc).__name__}: {exc}"})
                if len(failures) > max_failures:
                    raise RuntimeError(f"{len(failures)} clips failed to load; first: {failures[0]}") from exc
                continue
            if wave.ndim > 1:
                wave = wave.mean(axis=1)
            clips.append({"waveform": np.ascontiguousarray(wave, dtype=np.float32), "sample_rate": int(sr),
                          "label": label, "raw_label": row.raw_label, "speaker_id": "unknown",
                          "utterance_id": row.member, "attack": "bonafide" if label == BONAFIDE else "unknown"})
            loaded += 1
    report = {**info, "requested_per_class": n_per_class, "loaded": len(clips),
              "loaded_bonafide": sum(c["label"] == BONAFIDE for c in clips),
              "loaded_spoof": sum(c["label"] == SPOOF for c in clips),
              "failed": len(failures), "failures": failures[:20]}
    return clips, report


def write_synthetic_archive(path, n_real=4, n_fake=5, include_broken=True, sample_rate=TARGET_SR):
    """In-the-Wild-shaped zip (release_in_the_wild/*.wav + meta.csv with file/speaker/label) for the
    local smoke test and unit tests -- exercises the same fsspec zip path as the real hf:// archive.
    `include_broken` adds one undecodable member so failure recording is exercised too."""
    import soundfile as sf
    import zipfile

    rows = ["file,speaker,label"]
    rng = np.random.default_rng(0)
    with zipfile.ZipFile(path, "w") as zf:
        for i in range(n_real + n_fake):
            label = "bona-fide" if i < n_real else "spoof"
            t = np.arange(int(1.5 * sample_rate)) / sample_rate
            wave = (0.2 * np.sin(2 * np.pi * rng.uniform(100, 300) * t)).astype(np.float32)
            buf = io.BytesIO()
            sf.write(buf, wave, sample_rate, format="WAV")
            zf.writestr(f"release_in_the_wild/{i}.wav", buf.getvalue())
            rows.append(f"{i}.wav,Speaker {i % 3},{label}")
        if include_broken:
            zf.writestr("release_in_the_wild/broken.wav", b"not audio")
            rows.append("broken.wav,Speaker 0,spoof")
        zf.writestr("release_in_the_wild/meta.csv", "\n".join(rows))
    return str(path)


def write_synthetic_protocol(clips, path):
    """Protocol file in the official format, for the smoke test's protocol-verification path."""
    with open(path, "w", encoding="utf-8") as handle:
        for clip in clips:
            attack = "-" if clip["label"] == BONAFIDE else clip["attack"]
            key = "bonafide" if clip["label"] == BONAFIDE else "spoof"
            handle.write(f"{clip['speaker_id']} {clip['utterance_id']} - {attack} {key}\n")
    return str(path)


def make_synthetic_asvspoof(n_clips=120, seed=0, sample_rate=TARGET_SR, n_attacks=6, bonafide_pad_s=(0.02, 0.1)):
    """Tiny ASVspoof-shaped set for local smoke tests only -- never used for reported numbers.

    Deliberately includes the failure mode the real data is suspected of: spoof clips get more
    leading/trailing silence than bonafide ones, so the silence-leakage check has something to find.
    Spoof clips get one of `n_attacks` synthetic attack IDs, drawn independently of speaker (as in real
    ASVspoof, where every target speaker is spoofed by every system), from a separate RNG so adding
    attacks didn't change the synthetic audio other tests rely on.
    """
    rng = np.random.default_rng(seed)
    attack_rng = np.random.default_rng(seed + 1)
    clips = []
    for i in range(n_clips):
        label = BONAFIDE if i % 4 == 0 else SPOOF  # ~25% bonafide, imbalanced like LA
        attack = "bonafide" if label == BONAFIDE else f"A{7 + int(attack_rng.integers(n_attacks)):02d}"
        speech_s = rng.uniform(1.0, 2.5)
        t = np.arange(int(speech_s * sample_rate), dtype=np.float32) / sample_rate
        f0 = rng.uniform(100, 200)
        speech = 0.3 * np.sin(2 * np.pi * f0 * t) + 0.1 * np.sin(2 * np.pi * 2 * f0 * t)
        if label == SPOOF:  # a crude high-frequency "vocoder artifact"
            speech += 0.05 * np.sin(2 * np.pi * 6500 * t)
        speech += 0.01 * rng.standard_normal(speech.shape).astype(np.float32)
        pad_s = rng.uniform(0.4, 0.8) if label == SPOOF else rng.uniform(*bonafide_pad_s)
        pad = 1e-4 * rng.standard_normal(int(pad_s * sample_rate)).astype(np.float32)
        clips.append({
            "waveform": np.concatenate([pad, speech, pad]).astype(np.float32),
            "sample_rate": sample_rate,
            "label": label,
            "speaker_id": f"SPK_{i % 12:02d}",
            "utterance_id": f"SYN_{i:04d}",
            "path": f"synthetic/{i}.wav",
            "attack": attack,
        })
    return clips


def class_counts(clips):
    labels = np.array([c["label"] for c in clips])
    return {"bonafide": int((labels == BONAFIDE).sum()), "spoof": int((labels == SPOOF).sum()),
            "total": int(len(labels))}


def pos_weight(clips):
    """Weight for the positive (bonafide, minority) class in BCE, as in the graph engine."""
    counts = class_counts(clips)
    return float(counts["spoof"] / max(counts["bonafide"], 1))


def _both_classes_present(clips, indices):
    labels = {clips[i]["label"] for i in indices}
    return BONAFIDE in labels and SPOOF in labels


def speaker_disjoint_split(clips, test_fraction=0.3, seed=42, attempts=12):
    """Split so no speaker appears in both train and test.

    A random clip-level split lets the same speaker land on both sides, and a model can then key on
    speaker identity/recording channel instead of spoofing artifacts -- the audio analogue of the
    Phase 1 random-vs-temporal split leak. Speakers are shuffled and sent whole to the test side
    until it reaches `test_fraction`, so a speaker is never split across sides.

    Caveat this does NOT fix: every clip here comes from the LA *evaluation* partition, so the same
    spoofing systems (A07-A19) appear on both sides. Unseen-attack generalization is not measured.
    """
    by_speaker = {}
    for idx, clip in enumerate(clips):
        by_speaker.setdefault(clip["speaker_id"], []).append(idx)

    target_test = test_fraction * len(clips)
    for attempt in range(attempts):
        rng = np.random.default_rng(seed + attempt)
        speakers = sorted(by_speaker)
        rng.shuffle(speakers)

        train_idx, test_idx, n_test = [], [], 0
        for speaker in speakers:
            members = by_speaker[speaker]
            # Send speakers to test until adding another would overshoot the target by more than it undershoots.
            if n_test + len(members) / 2 <= target_test:
                test_idx.extend(members)
                n_test += len(members)
            else:
                train_idx.extend(members)
        # Both sides need both classes, or EER is undefined on that side. Speakers are assigned whole,
        # so this isn't guaranteed by construction -- reshuffle and retry.
        if train_idx and test_idx and _both_classes_present(clips, train_idx) and _both_classes_present(clips, test_idx):
            return sorted(train_idx), sorted(test_idx)

    # Too few speakers (or a speaker holds an entire class) for a disjoint split with both classes on
    # both sides. Fall back to a random split rather than returning an unusable one; the caller's
    # split report will show shared_speakers > 0, making the fallback visible instead of silent.
    return random_split(clips, test_fraction, seed)


def random_split(clips, test_fraction=0.3, seed=42):
    """Clip-level random split -- speakers appear on both sides. Kept as the leakage comparison."""
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(clips))
    n_test = int(round(test_fraction * len(clips)))
    return sorted(order[n_test:].tolist()), sorted(order[:n_test].tolist())


def split_report(clips, train_idx, test_idx):
    train_speakers = {clips[i]["speaker_id"] for i in train_idx}
    test_speakers = {clips[i]["speaker_id"] for i in test_idx}
    return {
        "train": class_counts([clips[i] for i in train_idx]),
        "test": class_counts([clips[i] for i in test_idx]),
        "train_speakers": len(train_speakers),
        "test_speakers": len(test_speakers),
        "shared_speakers": len(train_speakers & test_speakers),
    }

In [ ]:
# ---- inlined from engines/voice/preprocess.py (edit the module, then rebuild) ----
"""Audio front-end: resample -> silence trim -> fixed length -> LFCC (+deltas), plus the silence
ablations and training-time augmentation used in Phase 2B.

**Why LFCC, not mel.** LFCC is the front-end of the official ASVspoof 2019/2021 baselines. Its
filters are spaced linearly, so it keeps resolution at high frequencies, which is where vocoder and
TTS artifacts live. Mel spacing compresses exactly that region because it is tuned to human
perception, which is the right trade for ASR and the wrong one for spoof detection.

LFCC is implemented here in plain torch (linear triangular filterbank -> log -> DCT-II -> deltas)
rather than via `torchaudio.transforms.LFCC`, so the pipeline has no torchaudio dependency.
`LFCCExtractor` is batched and runs on any device; `VoicePreprocessor` uses the same extractor for
single clips, so cached features and on-the-fly (augmented) features are the same computation.
"""
import math

import numpy as np
import torch
import torch.nn.functional as F

TARGET_SR = 16000


def resample_linear(waveform, sr, target_sr=TARGET_SR):
    """Linear-interpolation resample. ASVspoof LA is already 16 kHz, so this is normally a no-op;
    it exists so an odd-rate out-of-domain clip doesn't silently go through at the wrong rate."""
    if sr == target_sr:
        return waveform, sr
    n_out = int(round(len(waveform) * target_sr / sr))
    x_in = np.linspace(0.0, 1.0, num=len(waveform), endpoint=False)
    x_out = np.linspace(0.0, 1.0, num=n_out, endpoint=False)
    return np.interp(x_out, x_in, waveform).astype(np.float32), target_sr


def frame_energy_db(waveform, sr, frame_ms=20.0, hop_ms=10.0):
    frame, hop = int(sr * frame_ms / 1000), int(sr * hop_ms / 1000)
    if len(waveform) < frame:
        waveform = np.pad(waveform, (0, frame - len(waveform)))
    frames = np.lib.stride_tricks.sliding_window_view(waveform, frame)[::hop]
    rms = np.sqrt(np.mean(frames.astype(np.float64) ** 2, axis=1) + 1e-12)
    return 20.0 * np.log10(rms + 1e-12), frame, hop


def trim_silence(waveform, sr, top_db=30.0, frame_ms=20.0, hop_ms=10.0):
    """Energy-based leading/trailing silence trim (librosa.effects.trim-style, no dependency).

    Returns (trimmed waveform, stats). Stats carry the leading/trailing silence and speech durations
    (what the silence-leakage probe measures) and the speech boundaries in samples (what the silence
    ablations cut at).
    """
    db, frame, hop = frame_energy_db(waveform, sr, frame_ms, hop_ms)
    voiced = np.flatnonzero(db > db.max() - top_db)
    total_s = len(waveform) / sr
    if len(voiced) == 0:  # nothing above the threshold: keep the clip, report it as all silence
        return waveform, {"total_s": total_s, "lead_silence_s": 0.0, "trail_silence_s": 0.0,
                          "speech_s": 0.0, "silence_ratio": 1.0, "all_silence": True,
                          "start_sample": 0, "end_sample": len(waveform)}
    start = int(voiced[0] * hop)
    end = min(len(waveform), int(voiced[-1] * hop + frame))
    trimmed = waveform[start:end]
    lead_s, trail_s = start / sr, (len(waveform) - end) / sr
    return trimmed, {
        "total_s": total_s,
        "lead_silence_s": lead_s,
        "trail_silence_s": trail_s,
        "speech_s": len(trimmed) / sr,
        "silence_ratio": (lead_s + trail_s) / total_s if total_s else 0.0,
        "all_silence": False,
        "start_sample": start,
        "end_sample": end,
    }


def fix_length(waveform, target_len):
    """Crop or repeat-pad to a fixed sample count (repeat-padding is the usual ASVspoof practice --
    zero-padding would inject exactly the silence cue this pipeline is trying to remove)."""
    if len(waveform) == 0:
        return np.zeros(target_len, dtype=np.float32)
    if len(waveform) >= target_len:
        return waveform[:target_len]
    repeats = int(np.ceil(target_len / len(waveform)))
    return np.tile(waveform, repeats)[:target_len].astype(np.float32)


# ---------------------------------------------------------------------------------------------------
# Silence ablations (Phase 2B, Task 4). Each variant removes a different part of what the silence
# could be leaking, so comparing them says *which* part the untrimmed model was using:
#
#   variant                    silence duration   silence content (what's actually in it)
#   untrimmed                        kept              kept
#   zeros_original_duration          kept              removed (digital zeros)
#   zeros_fixed / noise_fixed        removed           removed (identical zeros / identical-level noise)
#   silence_only (fixed window)      removed           kept -- and nothing else is kept
#   trimmed                          removed           removed
# ---------------------------------------------------------------------------------------------------
SILENCE_ABLATIONS = ("zeros_fixed", "noise_fixed", "zeros_original_duration")


def ablate_silence(waveform, sr, mode, stats, fixed_seconds=0.5, noise_dbfs=-60.0, rng=None):
    """Rebuild a clip with its leading and trailing silence replaced according to `mode`.

    Both ends are replaced, not just the leading one: trailing silence also leaks the label (it is
    the second-strongest timing cue), so replacing only the lead would leave a known cue in place and
    confound the test.
    """
    start, end = stats["start_sample"], stats["end_sample"]
    speech = waveform[start:end]
    if mode == "zeros_fixed":
        pad = np.zeros(int(fixed_seconds * sr), dtype=np.float32)
        return np.concatenate([pad, speech, pad])
    if mode == "noise_fixed":
        n = int(fixed_seconds * sr)
        rng = rng if rng is not None else np.random.default_rng(0)
        noise = (rng.standard_normal(2 * n) * 10 ** (noise_dbfs / 20)).astype(np.float32)
        return np.concatenate([noise[:n], speech, noise[n:]])
    if mode == "zeros_original_duration":
        return np.concatenate([np.zeros(start, dtype=np.float32), speech,
                               np.zeros(len(waveform) - end, dtype=np.float32)])
    raise ValueError(f"unknown silence ablation: {mode}")


def silence_window(waveform, stats, n_samples):
    """Exactly `n_samples` of the clip's genuine silence (leading, then trailing), or None if the clip
    has less silence than that. A fixed-size window means silence *duration* cannot reach the model;
    only what the silence contains can."""
    start, end = stats["start_sample"], stats["end_sample"]
    silence = np.concatenate([waveform[:start], waveform[end:]])
    if len(silence) < n_samples:
        return None
    return silence[:n_samples].astype(np.float32)


# ---------------------------------------------------------------------------------------------------
# LFCC
# ---------------------------------------------------------------------------------------------------
def linear_filterbank(n_filters, n_fft, sr):
    """Triangular filters spaced linearly in Hz (the 'linear' in LFCC)."""
    edges = torch.linspace(0.0, sr / 2.0, n_filters + 2)
    bin_freqs = torch.linspace(0.0, sr / 2.0, n_fft // 2 + 1)
    bank = torch.zeros(n_filters, len(bin_freqs))
    for i in range(n_filters):
        low, center, high = edges[i], edges[i + 1], edges[i + 2]
        rising = (bin_freqs - low) / torch.clamp(center - low, min=1e-6)
        falling = (high - bin_freqs) / torch.clamp(high - center, min=1e-6)
        bank[i] = torch.clamp(torch.minimum(rising, falling), min=0.0)
    return bank


def dct_matrix(n_out, n_in):
    """Orthonormal DCT-II."""
    n = torch.arange(n_in, dtype=torch.float32)
    k = torch.arange(n_out, dtype=torch.float32).unsqueeze(1)
    mat = torch.cos(torch.pi * k * (2 * n + 1) / (2 * n_in)) * np.sqrt(2.0 / n_in)
    mat[0] = mat[0] / np.sqrt(2.0)
    return mat


def deltas(features, width=2):
    """Standard regression deltas over a +/-`width` frame window, edges replicated. Accepts
    [coeffs, frames] or [batch, coeffs, frames]."""
    squeeze = features.dim() == 2
    x = features.unsqueeze(0) if squeeze else features
    padded = F.pad(x, (width, width), mode="replicate")
    n_frames = x.shape[-1]
    out = torch.zeros_like(x)
    for n in range(1, width + 1):
        out += n * (padded[..., width + n: width + n + n_frames] - padded[..., width - n: width - n + n_frames])
    out = out / (2.0 * sum(n * n for n in range(1, width + 1)))
    return out.squeeze(0) if squeeze else out


class LFCCExtractor(torch.nn.Module):
    """Batched LFCC (+deltas): [batch, samples] -> [batch, n_coeffs*3, frames], on any device.

    `filterbank_augment`, if given, is applied to the log linear-filterbank energies before the DCT.
    That is where frequency masking means something (masking whole frequency bands); masking rows
    of the cepstrum afterwards would not correspond to any frequency region.
    """

    def __init__(self, sample_rate=TARGET_SR, n_fft=512, win_ms=20.0, hop_ms=10.0, n_filters=70,
                 n_coeffs=20, add_deltas=True):
        super().__init__()
        self.n_fft = n_fft
        self.win_length = int(sample_rate * win_ms / 1000)
        self.hop_length = int(sample_rate * hop_ms / 1000)
        self.n_coeffs = n_coeffs
        self.add_deltas = add_deltas
        self.register_buffer("filterbank", linear_filterbank(n_filters, n_fft, sample_rate))
        self.register_buffer("dct", dct_matrix(n_coeffs, n_filters))
        self.register_buffer("window", torch.hann_window(self.win_length))

    @property
    def n_features(self):
        return self.n_coeffs * (3 if self.add_deltas else 1)

    def n_frames(self, n_samples):
        return n_samples // self.hop_length + 1

    def forward(self, waves, filterbank_augment=None):
        spec = torch.stft(waves, n_fft=self.n_fft, hop_length=self.hop_length, win_length=self.win_length,
                          window=self.window, center=True, pad_mode="reflect", return_complex=True)
        log_fbank = torch.log(torch.matmul(self.filterbank, spec.abs() ** 2) + 1e-10)
        if filterbank_augment is not None:
            log_fbank = filterbank_augment(log_fbank)
        feats = torch.matmul(self.dct, log_fbank)
        if self.add_deltas:
            d1 = deltas(feats)
            feats = torch.cat([feats, d1, deltas(d1)], dim=1)
        return feats


class VoicePreprocessor:
    """waveform -> (n_coeffs*3, n_frames) LFCC+delta+delta-delta tensor.

    Defaults follow the ASVspoof LFCC baseline: 20 ms window / 10 ms hop, 70 linear filters,
    20 cepstral coefficients, plus first and second deltas = 60 feature dimensions.
    """

    def __init__(self, sample_rate=TARGET_SR, clip_seconds=4.0, n_fft=512, win_ms=20.0, hop_ms=10.0,
                 n_filters=70, n_coeffs=20, trim=True, top_db=30.0, add_deltas=True):
        self.sample_rate = sample_rate
        self.clip_seconds = clip_seconds
        self.clip_samples = int(sample_rate * clip_seconds)
        self.trim = trim
        self.top_db = top_db
        self.extractor = LFCCExtractor(sample_rate, n_fft, win_ms, hop_ms, n_filters, n_coeffs, add_deltas)
        self.n_coeffs = n_coeffs
        self.add_deltas = add_deltas
        self.hop_length = self.extractor.hop_length

    @property
    def n_features(self):
        return self.extractor.n_features

    @property
    def n_frames(self):
        return self.extractor.n_frames(self.clip_samples)

    def prepare_waveform(self, waveform, sr):
        waveform, sr = resample_linear(np.asarray(waveform, dtype=np.float32), sr, self.sample_rate)
        stats = {"total_s": len(waveform) / sr, "lead_silence_s": 0.0, "trail_silence_s": 0.0,
                 "speech_s": len(waveform) / sr, "silence_ratio": 0.0, "all_silence": False,
                 "start_sample": 0, "end_sample": len(waveform)}
        if self.trim:
            waveform, stats = trim_silence(waveform, sr, self.top_db)
        return fix_length(waveform, self.clip_samples), stats

    @torch.no_grad()
    def lfcc(self, waveform):
        wave = torch.as_tensor(waveform, dtype=torch.float32).unsqueeze(0)
        return self.extractor(wave)[0]

    def __call__(self, waveform, sr):
        fixed, stats = self.prepare_waveform(waveform, sr)
        return self.lfcc(fixed), stats

    def featurize_clips(self, clips, progress_every=1000):
        """Returns (features [N,1,F,T] float32, labels [N], per-clip silence stats)."""
        features, labels, stats = [], [], []
        for i, clip in enumerate(clips):
            feats, clip_stats = self(clip["waveform"], clip["sample_rate"])
            features.append(feats)
            labels.append(clip["label"])
            stats.append(clip_stats)
            if progress_every and (i + 1) % progress_every == 0:
                print(f"  featurized {i + 1}/{len(clips)} clips")
        x = torch.stack(features).unsqueeze(1)
        return x, torch.tensor(labels, dtype=torch.long), stats


# ---------------------------------------------------------------------------------------------------
# Training-time augmentation (Phase 2B Run B). Applied to training batches only -- never to
# validation or test, which the data sources enforce by only calling these when `training=True`.
# ---------------------------------------------------------------------------------------------------
def simulate_low_bitrate(waves, generator, downsample_factor=4, quant_bits=6, mu=255.0, p_apply=0.3):
    """Cheap proxy for a low-bitrate lossy-codec (e.g. MP3) round-trip: linear-interpolation
    downsample/upsample (the bandwidth loss a low bitrate causes) plus mu-law companding quantized to
    `quant_bits` (the quantization noise a low bitrate causes). This is NOT a real MP3 encode/decode --
    no differentiable, per-batch-fast MP3 codec exists -- it is a documented approximation of the two
    dominant artifacts a low-bitrate codec introduces, applied to a random subset of clips per batch.
    waves: [batch, samples] on any device, expected in [-1, 1]."""
    batch, n = waves.shape
    device = waves.device
    apply = (torch.rand(batch, 1, generator=generator, device=device) < p_apply).float()

    narrow = F.interpolate(waves.unsqueeze(1), size=max(n // downsample_factor, 1), mode="linear",
                           align_corners=False)
    wide = F.interpolate(narrow, size=n, mode="linear", align_corners=False).squeeze(1)

    x = wide.clamp(-1.0, 1.0)
    companded = torch.sign(x) * torch.log1p(mu * x.abs()) / math.log1p(mu)
    levels = 2 ** quant_bits - 1
    quantized = torch.round((companded + 1) / 2 * levels) / levels * 2 - 1
    expanded = torch.sign(quantized) * (torch.expm1(quantized.abs() * math.log1p(mu)) / mu)

    return waves + apply * (expanded - waves)


def augment_waveforms(waves, generator, snr_db=(10.0, 40.0), gain_db=(-6.0, 6.0), p_noise=0.5,
                      codec_sim=False):
    """Additive white noise at a random per-clip SNR (applied to a random half of clips) and a random
    per-clip gain. waves: [batch, samples] on any device. `codec_sim=True` (Phase 2C Run F only) adds
    `simulate_low_bitrate` on top -- Phase 2B explicitly skipped a codec augmentation for time."""
    batch = waves.shape[0]
    device = waves.device

    def uniform(lo, hi):
        return lo + (hi - lo) * torch.rand(batch, 1, generator=generator, device=device)

    signal_power = waves.pow(2).mean(dim=1, keepdim=True).clamp_min(1e-10)
    noise_power = signal_power / 10 ** (uniform(*snr_db) / 10)
    noise = torch.randn(waves.shape, generator=generator, device=device) * noise_power.sqrt()
    apply = (torch.rand(batch, 1, generator=generator, device=device) < p_noise).float()
    waves = waves + apply * noise
    waves = waves * 10 ** (uniform(*gain_db) / 20)
    if codec_sim:
        waves = simulate_low_bitrate(waves, generator)
    return waves


def spec_augment(generator, n_freq_masks=2, max_freq_width=8, n_time_masks=2, max_time_width=40):
    """SpecAugment-style masking for the log-filterbank tensor [batch, filters, frames]. Masked
    regions are set to each clip's mean log energy, so masking doesn't inject an artificial floor."""

    def apply(log_fbank):
        batch, n_filters, n_frames = log_fbank.shape
        device = log_fbank.device
        out = log_fbank.clone()
        fill = log_fbank.mean(dim=(1, 2), keepdim=True).expand_as(out)
        filters = torch.arange(n_filters, device=device).view(1, -1, 1)
        frames = torch.arange(n_frames, device=device).view(1, 1, -1)
        mask = torch.zeros_like(out, dtype=torch.bool)
        for _ in range(n_freq_masks):
            width = torch.randint(0, max_freq_width + 1, (batch, 1, 1), generator=generator, device=device)
            start = (torch.rand(batch, 1, 1, generator=generator, device=device) * (n_filters - width).clamp_min(1)).long()
            mask |= (filters >= start) & (filters < start + width)
        for _ in range(n_time_masks):
            width = torch.randint(0, max_time_width + 1, (batch, 1, 1), generator=generator, device=device)
            start = (torch.rand(batch, 1, 1, generator=generator, device=device) * (n_frames - width).clamp_min(1)).long()
            mask |= (frames >= start) & (frames < start + width)
        return torch.where(mask, fill, out)

    return apply

In [ ]:
# ---- inlined from engines/voice/evaluate.py (edit the module, then rebuild) ----
"""EER, DET curve and the silence-leakage probe.

Convention throughout: score = "how bonafide is this", label 1 = bonafide (the target class).
  FAR (false accept rate) = spoof clips scored above threshold  -> accepted as bonafide
  FRR (false reject rate)  = bonafide clips scored below threshold -> rejected as spoof
EER is where those two curves cross, found by linear interpolation between the two bracketing
operating points rather than by taking the nearest point, which is only an approximation.
"""
import numpy as np
from sklearn.metrics import roc_curve


def compute_eer(scores, labels, positive_label=1):
    """Equal Error Rate. Returns eer, the threshold at it, and FAR/FRR there."""
    scores = np.asarray(scores, dtype=np.float64)
    labels = np.asarray(labels)
    y = (labels == positive_label).astype(int)
    if y.sum() == 0 or y.sum() == len(y):
        return {"eer": float("nan"), "threshold": float("nan"), "far": float("nan"),
                "frr": float("nan"), "note": "only one class present; EER undefined"}

    fpr, tpr, thresholds = roc_curve(y, scores)
    far, frr = fpr, 1.0 - tpr  # FAR rises with threshold-looseness, FRR falls
    diff = far - frr
    sign_change = np.flatnonzero(np.sign(diff[:-1]) * np.sign(diff[1:]) < 0)
    if len(sign_change) == 0:  # curves touch rather than cross (ties/small samples)
        idx = int(np.nanargmin(np.abs(diff)))
        return {"eer": float((far[idx] + frr[idx]) / 2), "threshold": float(thresholds[idx]),
                "far": float(far[idx]), "frr": float(frr[idx]), "interpolated": False}
    i = int(sign_change[0])
    # Linear interpolation of the crossing between operating points i and i+1.
    weight = diff[i] / (diff[i] - diff[i + 1])
    eer = float(far[i] + weight * (far[i + 1] - far[i]))
    frr_at = float(frr[i] + weight * (frr[i + 1] - frr[i]))
    threshold = float(thresholds[i] + weight * (thresholds[i + 1] - thresholds[i]))
    return {"eer": (eer + frr_at) / 2, "threshold": threshold, "far": eer, "frr": frr_at,
            "interpolated": True}


def det_curve_points(scores, labels, positive_label=1, max_points=512):
    """FAR/FRR pairs for plotting a DET curve, thinned to at most `max_points`."""
    y = (np.asarray(labels) == positive_label).astype(int)
    fpr, tpr, thresholds = roc_curve(y, np.asarray(scores, dtype=np.float64))
    far, frr = fpr, 1.0 - tpr
    if len(far) > max_points:
        keep = np.unique(np.linspace(0, len(far) - 1, max_points).astype(int))
        far, frr, thresholds = far[keep], frr[keep], thresholds[keep]
    return {"far": far.tolist(), "frr": frr.tolist(), "thresholds": thresholds.tolist()}


def per_class_accuracy(scores, labels, threshold, positive_label=1):
    scores, labels = np.asarray(scores), np.asarray(labels)
    predicted_positive = scores > threshold
    is_positive = labels == positive_label
    return {
        "threshold": float(threshold),
        "bonafide_accuracy": float(predicted_positive[is_positive].mean()) if is_positive.any() else None,
        "spoof_accuracy": float((~predicted_positive[~is_positive]).mean()) if (~is_positive).any() else None,
        "overall_accuracy": float((predicted_positive == is_positive).mean()),
        "n_bonafide": int(is_positive.sum()),
        "n_spoof": int((~is_positive).sum()),
    }


def _stratified_resample(labels, rng, positive_label=1):
    """Resample with replacement *within* each class, so every resample keeps the original class
    counts -- a plain resample could drop the (small) bonafide class and leave EER undefined."""
    pos = np.flatnonzero(labels == positive_label)
    neg = np.flatnonzero(labels != positive_label)
    return np.concatenate([rng.choice(pos, len(pos)), rng.choice(neg, len(neg))])


def bootstrap_eer_ci(scores, labels, n_resamples=1000, seed=0, alpha=0.05, positive_label=1):
    """Percentile bootstrap CI for EER, stratified by class."""
    scores, labels = np.asarray(scores, dtype=np.float64), np.asarray(labels)
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(n_resamples):
        idx = _stratified_resample(labels, rng, positive_label)
        samples.append(compute_eer(scores[idx], labels[idx], positive_label)["eer"])
    samples = np.array(samples)
    return {
        "eer": compute_eer(scores, labels, positive_label)["eer"],
        "ci_low": float(np.quantile(samples, alpha / 2)),
        "ci_high": float(np.quantile(samples, 1 - alpha / 2)),
        "ci_width": float(np.quantile(samples, 1 - alpha / 2) - np.quantile(samples, alpha / 2)),
        "bootstrap_std": float(samples.std()),
        "n_resamples": n_resamples,
        "n_bonafide": int((labels == positive_label).sum()),
        "n_spoof": int((labels != positive_label).sum()),
    }


def bootstrap_ci_at_counts(scores, labels, n_bonafide, n_spoof, n_resamples=1000, seed=0, alpha=0.05,
                           positive_label=1):
    """CI width the same scores would have had with a smaller test set of the given class counts.
    Used to say how much the larger Phase 2B test set narrowed the CI versus Phase 2's, on the same
    model, instead of comparing CIs from different models."""
    scores, labels = np.asarray(scores, dtype=np.float64), np.asarray(labels)
    pos = np.flatnonzero(labels == positive_label)
    neg = np.flatnonzero(labels != positive_label)
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(n_resamples):
        idx = np.concatenate([rng.choice(pos, n_bonafide), rng.choice(neg, n_spoof)])
        samples.append(compute_eer(scores[idx], labels[idx], positive_label)["eer"])
    low, high = np.quantile(samples, [alpha / 2, 1 - alpha / 2])
    return {"n_bonafide": int(n_bonafide), "n_spoof": int(n_spoof), "ci_low": float(low),
            "ci_high": float(high), "ci_width": float(high - low)}


def bootstrap_eer_difference(scores_a, labels_a, scores_b, labels_b, n_resamples=1000, seed=0,
                             alpha=0.05, paired=False, positive_label=1):
    """Percentile bootstrap CI for EER(a) - EER(b).

    paired=True: a and b are two models scored on the *same* clips in the same order, so each resample
    draws one set of indices used for both -- this cancels the test-set sampling noise the two share.
    paired=False: two different test sets (e.g. a random split vs. a speaker-disjoint split), each
    resampled independently.
    """
    sa, la = np.asarray(scores_a, dtype=np.float64), np.asarray(labels_a)
    sb, lb = np.asarray(scores_b, dtype=np.float64), np.asarray(labels_b)
    if paired and (len(sa) != len(sb) or not np.array_equal(la, lb)):
        raise ValueError("paired bootstrap needs both models scored on the same clips in the same order")
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_resamples):
        ia = _stratified_resample(la, rng, positive_label)
        ib = ia if paired else _stratified_resample(lb, rng, positive_label)
        diffs.append(compute_eer(sa[ia], la[ia], positive_label)["eer"] -
                     compute_eer(sb[ib], lb[ib], positive_label)["eer"])
    low, high = np.quantile(diffs, [alpha / 2, 1 - alpha / 2])
    point = compute_eer(sa, la, positive_label)["eer"] - compute_eer(sb, lb, positive_label)["eer"]
    return {"difference": float(point), "ci_low": float(low), "ci_high": float(high),
            "excludes_zero": bool(low > 0 or high < 0), "paired": paired, "n_resamples": n_resamples}


def speaker_leakage_call(diff):
    """Phase 2 printed "speaker leakage" for any positive difference, including +0.0009. The rule is
    now: only a CI that excludes zero counts. `diff` = EER(speaker-disjoint) - EER(random split), so a
    positive, significant difference means the random split flatters the model."""
    ci = f"{diff['difference']:+.4f} EER, 95% CI [{diff['ci_low']:+.4f}, {diff['ci_high']:+.4f}]"
    if not diff["excludes_zero"]:
        return {"verdict": "not_detectable",
                "text": f"no leakage detectable at this sample size ({ci}; the CI includes zero)"}
    if diff["difference"] > 0:
        return {"verdict": "leakage_detected",
                "text": f"speaker leakage detected: the random split looks better ({ci}, CI excludes zero)"}
    return {"verdict": "random_split_worse",
            "text": f"the random split is significantly *worse* ({ci}) -- not leakage; investigate the splits"}


def per_attack_eer(scores, labels, attacks, positive_label=1):
    """EER of all bonafide clips against each spoofing attack separately."""
    scores, labels, attacks = np.asarray(scores), np.asarray(labels), np.asarray(attacks)
    bonafide = labels == positive_label
    out = {}
    for attack in sorted(set(attacks[~bonafide].tolist())):
        mask = bonafide | (attacks == attack)
        out[attack] = {"eer": compute_eer(scores[mask], labels[mask], positive_label)["eer"],
                       "n_spoof": int((attacks == attack).sum())}
    return out


def evaluate_scores(scores, labels, positive_label=1):
    eer = compute_eer(scores, labels, positive_label)
    return {
        **eer,
        "per_class_at_eer_threshold": per_class_accuracy(scores, labels, eer["threshold"], positive_label),
        "det_curve": det_curve_points(scores, labels, positive_label),
    }


def duration_leakage_probe(stats, labels, positive_label=1):
    """How much can clip timing alone tell you about the label?

    Scores each clip by a duration measure and computes the EER of that single feature. An EER far
    from 50% means the dataset's recording/segmentation artifacts leak the label, independent of any
    spoofing artifact -- the known ASVspoof silence-duration shortcut. Both directions are tried per
    measure, since which sign favours bonafide is itself an empirical question; the reported EER is
    the better (lower) of the two, i.e. the strongest cheat available from that measure.
    """
    labels = np.asarray(labels)
    out = {}
    for key in ("total_s", "lead_silence_s", "trail_silence_s", "speech_s", "silence_ratio"):
        values = np.array([s[key] for s in stats], dtype=np.float64)
        if np.allclose(values, values[0]):
            out[key] = {"eer": float("nan"), "note": "constant measure"}
            continue
        forward = compute_eer(values, labels, positive_label)["eer"]
        backward = compute_eer(-values, labels, positive_label)["eer"]
        better = min(forward, backward)
        y = (labels == positive_label).astype(float)
        out[key] = {
            "eer": float(better),
            "direction": "higher_is_bonafide" if forward <= backward else "lower_is_bonafide",
            "point_biserial_r": float(np.corrcoef(values, y)[0, 1]),
            "mean_bonafide": float(values[labels == positive_label].mean()),
            "mean_spoof": float(values[labels != positive_label].mean()),
        }
    return out


def leakage_verdict(probe, strong=0.35, mild=0.45):
    """Turn the probe into a stated finding. `strong`/`mild` are EER thresholds: the further below
    50% a single duration measure gets, the more label information timing alone carries."""
    usable = {k: v["eer"] for k, v in probe.items() if not np.isnan(v["eer"])}
    if not usable:
        return {"level": "unknown", "worst_measure": None, "worst_eer": None,
                "text": "No usable duration measure; leakage could not be assessed."}
    worst_measure = min(usable, key=usable.get)
    worst = usable[worst_measure]
    level = "strong" if worst < strong else "mild" if worst < mild else "none"
    text = {
        "strong": f"Timing alone separates the classes well ({worst_measure} EER {worst:.4f}), so this data "
                  f"carries the known ASVspoof silence-duration shortcut.",
        "mild": f"Timing alone is weakly predictive ({worst_measure} EER {worst:.4f}) -- some leakage, but far "
                f"from a free solution.",
        "none": f"No meaningful timing leakage: the best duration measure only reaches EER {worst:.4f} "
                f"({worst_measure}), close to the 0.5 of a coin flip.",
    }[level]
    return {"level": level, "worst_measure": worst_measure, "worst_eer": float(worst), "text": text}

In [ ]:
# ---- inlined from engines/voice/model.py (edit the module, then rebuild) ----
"""LCNN (Light CNN) spoof detector, the SSL layer-weighted head, and a shared training loop.

LCNN is the CNN architecture used by the ASVspoof baselines. Its defining piece is the
Max-Feature-Map (MFM) activation: a conv produces 2k channels, which are split in half and combined
with an element-wise max, halving the channels. That acts as a learned feature selector and keeps
the network small.

Every model outputs one logit per clip, where a higher score means "more bonafide" (label 1 =
bonafide), matching the EER convention in evaluate.py.
"""
import math

import numpy as np
import torch
import torch.nn.functional as F



class MFM(torch.nn.Module):
    """Max-Feature-Map over channels (dim 1)."""

    def forward(self, x):
        a, b = torch.chunk(x, 2, dim=1)
        return torch.maximum(a, b)


def conv_mfm(in_ch, out_ch, kernel_size, padding):
    return torch.nn.Sequential(
        torch.nn.Conv2d(in_ch, out_ch * 2, kernel_size=kernel_size, padding=padding),
        MFM(),
    )


class LCNN(torch.nn.Module):
    MIN_SIZE = 16  # four 2x max-pools: each input axis must be at least 2**4 long

    def __init__(self, n_features=60, n_frames=401, channels=(16, 24, 32, 32), dropout=0.3, fc_dim=80):
        super().__init__()
        if n_features < self.MIN_SIZE or n_frames < self.MIN_SIZE:
            raise ValueError(f"LCNN needs at least {self.MIN_SIZE} feature rows and frames, got "
                             f"{n_features}x{n_frames} -- use a longer input window")
        c1, c2, c3, c4 = channels
        self.blocks = torch.nn.Sequential(
            conv_mfm(1, c1, 5, 2),
            torch.nn.MaxPool2d(2),
            conv_mfm(c1, c1, 1, 0),
            torch.nn.BatchNorm2d(c1),
            conv_mfm(c1, c2, 3, 1),
            torch.nn.MaxPool2d(2),
            torch.nn.BatchNorm2d(c2),
            conv_mfm(c2, c2, 1, 0),
            torch.nn.BatchNorm2d(c2),
            conv_mfm(c2, c3, 3, 1),
            torch.nn.MaxPool2d(2),
            conv_mfm(c3, c3, 1, 0),
            torch.nn.BatchNorm2d(c3),
            conv_mfm(c3, c4, 3, 1),
            torch.nn.MaxPool2d(2),
        )
        with torch.no_grad():  # measure the flattened size instead of deriving it from the pooling arithmetic
            flat_dim = self.blocks(torch.zeros(1, 1, n_features, n_frames)).numel()
        self.head = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(flat_dim, fc_dim * 2),
            MFM(),  # operates on dim 1, which is the feature dim for a 2D tensor
            torch.nn.BatchNorm1d(fc_dim),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(fc_dim, 1),
        )

    def forward(self, x):
        return self.head(self.blocks(x)).squeeze(-1)


class SSLHead(torch.nn.Module):
    """Classifier over pooled self-supervised encoder features [batch, layers, dim].

    A learned softmax weight per encoder layer mixes the layers into one vector, then a small MLP
    scores it. The layer weights are reported after training: they say which depth of the encoder
    carried the spoofing information.
    """

    def __init__(self, n_layers, feat_dim, hidden=128, dropout=0.3):
        super().__init__()
        self.layer_logits = torch.nn.Parameter(torch.zeros(n_layers))
        self.mlp = torch.nn.Sequential(
            torch.nn.LayerNorm(feat_dim),
            torch.nn.Linear(feat_dim, hidden),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden, 1),
        )

    def forward(self, x):
        weights = torch.softmax(self.layer_logits, dim=0)
        return self.mlp((weights.view(1, -1, 1) * x).sum(dim=1)).squeeze(-1)

    def layer_weights(self):
        return torch.softmax(self.layer_logits.detach(), dim=0).cpu().tolist()

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


class SSLSeqHead(torch.nn.Module):
    """Phase 2C Model E: a single-layer bidirectional LSTM over one frozen SSL layer's per-frame
    token sequence [batch, frames, hidden], mean-pooled over time, then a small MLP head.

    Only one SSL layer is used (picked from Model C's learned layer weights, logged in the notebook),
    so there is no per-layer weighting mechanism here -- unlike `SSLHead`, which mixes every layer.
    Kept deliberately small (BiLSTM hidden size 64 by default): the goal is to test whether *any*
    temporal modeling helps, not to add capacity as a confound. `n_added_params` lets the notebook log
    that confound directly against `SSLHead`'s parameter count.
    """

    def __init__(self, feat_dim, lstm_hidden=64, hidden=128, dropout=0.3):
        super().__init__()
        self.lstm = torch.nn.LSTM(feat_dim, lstm_hidden, num_layers=1, batch_first=True, bidirectional=True)
        out_dim = 2 * lstm_hidden
        self.mlp = torch.nn.Sequential(
            torch.nn.LayerNorm(out_dim),
            torch.nn.Linear(out_dim, hidden),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(hidden, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.mlp(out.mean(dim=1)).squeeze(-1)

    def n_params(self):
        return sum(p.numel() for p in self.parameters())


def batches(n, batch_size, shuffle, generator=None):
    order = torch.randperm(n, generator=generator) if shuffle else torch.arange(n)
    for start in range(0, n, batch_size):
        yield order[start:start + batch_size]


def score_model(model, fetch, indices, batch_size=64):
    """Scores (higher = more bonafide) for `indices`, in order. `fetch(idx, training=False)`."""
    model.eval()
    indices = np.asarray(indices)
    out = []
    with torch.no_grad():
        for start in range(0, len(indices), batch_size):
            out.append(model(fetch(indices[start:start + batch_size], False)).detach().float().cpu())
    return torch.cat(out).numpy().astype(np.float64) if out else np.zeros(0)


def train_model(model, fetch, labels, train_idx, pos_weight, device, val_idx=None, epochs=20,
                batch_size=32, lr=3e-4, weight_decay=1e-4, patience=5, seed=42, verbose=True):
    """Class-weighted BCE training with optional early stopping on validation EER.

    `fetch(idx, training)` returns the model input for clip indices `idx` on `device`; data sources
    apply augmentation only when `training` is True, so validation and test inputs are never
    augmented. `labels` is indexed by the same clip indices.
    """
    labels = torch.as_tensor(np.asarray(labels))
    train_idx = np.asarray(train_idx)
    generator = torch.Generator().manual_seed(seed)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    pos_weight_t = torch.tensor([pos_weight], device=device, dtype=torch.float32)

    history, best = [], {"eer": math.inf, "epoch": 0, "state": None}
    since_improve, val_unusable = 0, False
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, n_seen = 0.0, 0
        for positions in batches(len(train_idx), batch_size, shuffle=True, generator=generator):
            if len(positions) < 2:
                continue  # BatchNorm needs at least 2 samples
            idx = train_idx[positions.numpy()]
            xb = fetch(idx, True)
            yb = labels[idx].to(device).float()
            optimizer.zero_grad()
            loss = F.binary_cross_entropy_with_logits(model(xb), yb, pos_weight=pos_weight_t)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(idx)
            n_seen += len(idx)
        epoch_loss = total_loss / max(n_seen, 1)
        if not math.isfinite(epoch_loss):
            raise FloatingPointError(f"training diverged at epoch {epoch} (loss={epoch_loss})")

        entry = {"epoch": epoch, "train_loss": epoch_loss}
        if val_idx is not None and not val_unusable:
            entry["val_eer"] = compute_eer(score_model(model, fetch, val_idx), labels[np.asarray(val_idx)].numpy())["eer"]
            if not math.isfinite(entry["val_eer"]):
                # One-class validation set: EER is undefined, so model selection and early stopping
                # have nothing to act on. Say so loudly rather than silently keeping the last epoch.
                val_unusable = True
                print("  WARNING: validation EER is undefined (validation set has a single class) -- "
                      "disabling early stopping and best-epoch selection for this run")
            elif entry["val_eer"] < best["eer"] - 1e-6:
                best = {"eer": entry["val_eer"], "epoch": epoch,
                        "state": {k: v.detach().clone() for k, v in model.state_dict().items()}}
                since_improve = 0
            else:
                since_improve += 1
        history.append(entry)
        if verbose:
            print(f"  epoch {epoch:3d}  loss {epoch_loss:.4f}" +
                  (f"  val EER {entry['val_eer']:.4f}" if "val_eer" in entry else ""))
        if patience is not None and val_idx is not None and not val_unusable and since_improve >= patience:
            if verbose:
                print(f"  early stopping at epoch {epoch} (best val EER {best['eer']:.4f} at epoch {best['epoch']})")
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"])
    usable_val = val_idx is not None and not val_unusable
    return {"history": history, "best_val_eer": best["eer"] if usable_val else None,
            "best_epoch": best["epoch"] if usable_val else len(history),
            "epochs_run": len(history), "validation_usable": usable_val}


def train_lcnn(model, x_train, y_train, pos_weight, device, epochs=20, batch_size=32, lr=3e-4,
               weight_decay=1e-4, x_val=None, y_val=None, patience=5, seed=42, verbose=True):
    """In-memory wrapper around `train_model` (the Phase 2 notebook's interface)."""
    n_train = len(x_train)
    x_all = x_train if x_val is None else torch.cat([x_train, x_val])
    y_all = y_train if y_val is None else torch.cat([y_train, y_val])
    val_idx = None if x_val is None else np.arange(n_train, n_train + len(x_val))
    return train_model(model, lambda idx, training: x_all[torch.as_tensor(idx)].to(device), y_all,
                       np.arange(n_train), pos_weight, device, val_idx=val_idx, epochs=epochs,
                       batch_size=batch_size, lr=lr, weight_decay=weight_decay, patience=patience,
                       seed=seed, verbose=verbose)


@torch.no_grad()
def predict_scores(model, x, device, batch_size=64):
    """Scores where higher = more bonafide, for an in-memory feature tensor."""
    return score_model(model, lambda idx, training: x[torch.as_tensor(idx)].to(device),
                       np.arange(len(x)), batch_size)

In [ ]:
# ---- inlined from engines/voice/features.py (edit the module, then rebuild) ----
"""On-disk feature cache for Phase 2B, and the data sources training reads from.

One decode pass over the selected clips writes everything later steps need, as memory-mapped
arrays, so a kernel crash mid-run never forces the audio to be downloaded and decoded again:

  wave_trimmed.npy     int16 [N, samples]   trimmed, fixed-length waveforms. Runs A and B both read
                                            this and compute LFCC on the GPU per batch -- B only adds
                                            augmentation, so A vs. B changes exactly one variable. The
                                            SSL encoder (Run C) reads it too.
  lfcc_<variant>.npy   float16 [N, F, T]    untrimmed and silence-ablation features (comparisons only).
  lfcc_silence_only.npy                     LFCC of a fixed window of genuine silence per clip.
  clip_stats.json                           silence stats + which clips have enough silence.
  manifest.json                             what was built; a matching manifest means "reuse".

Every waveform is quantized to int16 before its LFCC is computed, so cached and on-the-fly
features see the same signal precision.
"""
import json
import time
from pathlib import Path

import numpy as np
import torch


INT16_SCALE = 32767.0
LFCC_VARIANTS = ("untrimmed",) + SILENCE_ABLATIONS


def to_int16(wave):
    return np.clip(np.round(wave * INT16_SCALE), -32768, 32767).astype(np.int16)


def quantize(wave):
    return to_int16(wave).astype(np.float32) / INT16_SCALE


def _manifest_matches(path, expected):
    if not path.exists():
        return False
    try:
        return json.loads(path.read_text(encoding="utf-8")).get("config") == expected
    except (json.JSONDecodeError, OSError):
        return False


def build_feature_cache(clip_iter, n_clips, cache_dir, extractor, device, clip_samples, silence_samples,
                        top_db=30.0, fixed_silence_s=0.5, noise_dbfs=-60.0, seed=42, batch_size=128,
                        config_key=None, progress_every=2000):
    """Single pass: decode each clip once, build every waveform variant, LFCC them on `device` in
    batches, and write the memmaps. Reuses an existing cache when its manifest matches."""
    from numpy.lib.format import open_memmap

    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = cache_dir / "manifest.json"
    expected = {"n_clips": int(n_clips), "clip_samples": int(clip_samples), "silence_samples": int(silence_samples),
                "top_db": top_db, "fixed_silence_s": fixed_silence_s, "noise_dbfs": noise_dbfs,
                "variants": list(LFCC_VARIANTS), "key": config_key}
    if _manifest_matches(manifest_path, expected):
        print(f"reusing feature cache in {cache_dir}")
        return open_feature_cache(cache_dir)

    n_feat = extractor.n_features
    frames, silence_frames = extractor.n_frames(clip_samples), extractor.n_frames(silence_samples)
    waves = open_memmap(cache_dir / "wave_trimmed.npy", mode="w+", dtype=np.int16, shape=(n_clips, clip_samples))
    lfcc = {v: open_memmap(cache_dir / f"lfcc_{v}.npy", mode="w+", dtype=np.float16, shape=(n_clips, n_feat, frames))
            for v in LFCC_VARIANTS}
    lfcc_silence = open_memmap(cache_dir / "lfcc_silence_only.npy", mode="w+", dtype=np.float16,
                               shape=(n_clips, n_feat, silence_frames))
    stats_out, has_silence = [], np.zeros(n_clips, dtype=bool)
    pending = {v: [] for v in LFCC_VARIANTS + ("silence_only",)}

    def flush(variant, target):
        rows = pending[variant]
        if not rows:
            return
        batch = torch.from_numpy(np.stack([w for _, w in rows])).to(device)
        with torch.no_grad():
            feats = extractor(batch).half().cpu().numpy()
        for (row, _), f in zip(rows, feats):
            target[row] = f
        pending[variant] = []

    start_time = time.time()
    count = 0
    for i, clip in enumerate(clip_iter):
        wave, sr = resample_linear(np.asarray(clip["waveform"], dtype=np.float32), clip["sample_rate"])
        _, stats = trim_silence(wave, sr, top_db)
        stats_out.append({k: stats[k] for k in ("total_s", "lead_silence_s", "trail_silence_s",
                                                  "speech_s", "silence_ratio", "all_silence")})
        waves[i] = to_int16(fix_length(wave[stats["start_sample"]:stats["end_sample"]], clip_samples))

        rng = np.random.default_rng(seed + i)
        variant_waves = {"untrimmed": fix_length(wave, clip_samples)}
        for mode in SILENCE_ABLATIONS:
            variant_waves[mode] = fix_length(ablate_silence(wave, sr, mode, stats, fixed_silence_s, noise_dbfs, rng),
                                             clip_samples)
        for variant, w in variant_waves.items():
            pending[variant].append((i, quantize(w)))
            if len(pending[variant]) >= batch_size:
                flush(variant, lfcc[variant])

        window = silence_window(wave, stats, silence_samples)
        has_silence[i] = window is not None
        pending["silence_only"].append((i, quantize(window if window is not None else np.zeros(silence_samples, np.float32))))
        if len(pending["silence_only"]) >= batch_size:
            flush("silence_only", lfcc_silence)

        count = i + 1
        if progress_every and count % progress_every == 0:
            rate = count / max(time.time() - start_time, 1e-6)
            print(f"  featurized {count}/{n_clips} clips ({rate:.0f} clips/s)")
    if count != n_clips:
        raise RuntimeError(f"expected {n_clips} clips, got {count}")
    for variant in LFCC_VARIANTS:
        flush(variant, lfcc[variant])
    flush("silence_only", lfcc_silence)

    for array in [waves, lfcc_silence, *lfcc.values()]:
        array.flush()
    (cache_dir / "clip_stats.json").write_text(
        json.dumps({"stats": stats_out, "has_silence_window": has_silence.tolist()}), encoding="utf-8")
    manifest_path.write_text(json.dumps({"config": expected, "built_seconds": round(time.time() - start_time, 1)},
                                        indent=2), encoding="utf-8")
    return open_feature_cache(cache_dir)


def open_feature_cache(cache_dir):
    cache_dir = Path(cache_dir)
    extra = json.loads((cache_dir / "clip_stats.json").read_text(encoding="utf-8"))
    return {
        "wave_trimmed": np.load(cache_dir / "wave_trimmed.npy", mmap_mode="r"),
        "lfcc": {v: np.load(cache_dir / f"lfcc_{v}.npy", mmap_mode="r") for v in LFCC_VARIANTS},
        "lfcc_silence_only": np.load(cache_dir / "lfcc_silence_only.npy", mmap_mode="r"),
        "stats": extra["stats"],
        "has_silence_window": np.array(extra["has_silence_window"], dtype=bool),
    }


def cache_size_gb(cache_dir):
    return sum(p.stat().st_size for p in Path(cache_dir).glob("*") if p.is_file()) / 1e9


# ---------------------------------------------------------------------------------------------------
# Data sources: fetch(idx, training) -> model input on `device`. Augmentation only when training.
# ---------------------------------------------------------------------------------------------------
def lfcc_array_source(array, device):
    """Cached float16 LFCC [N, F, T] -> LCNN input [B, 1, F, T]."""
    def fetch(idx, training):
        return torch.from_numpy(np.asarray(array[np.asarray(idx)], dtype=np.float32)).unsqueeze(1).to(device)
    return fetch


def wave_lfcc_source(waves_int16, extractor, device, augment=False, seed=0):
    """int16 waveforms -> (optionally augmented) LFCC on `device` -> LCNN input [B, 1, F, T]."""
    generator = torch.Generator(device=device).manual_seed(seed) if augment else None

    def fetch(idx, training):
        w = torch.from_numpy(np.asarray(waves_int16[np.asarray(idx)], dtype=np.float32) / INT16_SCALE).to(device)
        apply = augment and training
        if apply:
            w = augment_waveforms(w, generator)
        with torch.no_grad():
            feats = extractor(w, filterbank_augment=spec_augment(generator) if apply else None)
        return feats.unsqueeze(1)
    return fetch


def array_source(array, device):
    """Cached pooled SSL features [N, layers, dim] -> SSLHead input. Also used as-is for Phase 2C
    Model E's per-frame sequence cache [N, frames, hidden] -> SSLSeqHead input, since both are just a
    memmap slice on the leading axis."""
    def fetch(idx, training):
        return torch.from_numpy(np.asarray(array[np.asarray(idx)], dtype=np.float32)).to(device)
    return fetch


def array_mean_only_source(pooled_array, hidden_size, device):
    """Phase 2C Model D: the same cached mean+std pooled SSL features as Model C, sliced to the mean
    half only. No extra encoder pass -- `ssl.py`'s `pooled()` already concatenates mean and std, so
    this ablates Model D's one variable (std pooling) by reading less of the existing cache."""
    def fetch(idx, training):
        arr = np.asarray(pooled_array[np.asarray(idx)], dtype=np.float32)[..., :hidden_size]
        return torch.from_numpy(arr).to(device)
    return fetch


def wave_ssl_sequence_source(waves_int16, ssl_encoder, layer, device):
    """int16 waveforms -> on-the-fly per-frame sequence of one frozen SSL layer (Phase 2C Model E).

    Deliberately not cached to disk: a full-dataset single-layer sequence cache is tens of GB (see
    `ssl.py`'s `build_ssl_sequence_cache`), and stacking it on top of the feature cache this notebook
    already builds overran Kaggle's available disk on a real run (Version 2, 2026-09-29 -- issue #25).
    Same trade as `wave_ssl_source`: no disk cost, more GPU time re-running the frozen encoder.
    """
    def fetch(idx, training):
        w = torch.from_numpy(np.asarray(waves_int16[np.asarray(idx)], dtype=np.float32) / INT16_SCALE).to(device)
        return ssl_encoder.sequence(w, layer)
    return fetch


def wave_ssl_source(waves_int16, ssl_encoder, device, augment=False, seed=0, codec_sim=False, pooled_cache=None):
    """int16 waveforms -> on-the-fly frozen-SSL mean+std pooled features (Phase 2C Model F).

    Augmentation must happen upstream of the frozen encoder's forward pass, so -- unlike every other
    SSL run, which reads the pre-computed cache in `ssl.py` -- training calls (`training=True`) re-run
    the (frozen, no-grad) encoder every batch. Slower than the cache, but the encoder's own weights
    never see a gradient.

    `pooled_cache`, if given (Phase 2B's already-built `ssl_features`, same encoder/pooling/clips): used
    for every non-training call instead of re-running the encoder. Validation/test are never augmented,
    so their features are identical to that cache's -- recomputing them from the waveform every epoch
    (as an earlier version of this function did) was pure wasted GPU time, since validation runs once
    per epoch for up to `SSL_EPOCHS` epochs and test runs once more after training.
    """
    generator = torch.Generator(device=device).manual_seed(seed) if augment else None

    def fetch(idx, training):
        idx = np.asarray(idx)
        if not training and pooled_cache is not None:
            return torch.from_numpy(np.asarray(pooled_cache[idx], dtype=np.float32)).to(device)
        w = torch.from_numpy(np.asarray(waves_int16[idx], dtype=np.float32) / INT16_SCALE).to(device)
        if augment and training:
            w = augment_waveforms(w, generator, codec_sim=codec_sim)
        return ssl_encoder.pooled(w)
    return fetch

In [ ]:
# ---- inlined from engines/voice/ssl.py (edit the module, then rebuild) ----
"""Frozen self-supervised speech encoder (wav2vec 2.0 / XLS-R family) as a front-end (Phase 2B Run C).

The encoder is frozen and run once over every clip. Each layer's frame sequence is pooled to its
mean and standard deviation and cached, so the classifier (`model.SSLHead`, a learned softmax weight
per layer + a small MLP) trains in seconds and extra seeds cost almost nothing. Fine-tuning the
encoder is deliberately not done in this phase.
"""
from pathlib import Path

import numpy as np
import torch



class SSLEncoder:
    def __init__(self, model_name=None, device=torch.device("cpu"), fp16=True, model=None):
        if model is None:
            from transformers import AutoModel

            model = AutoModel.from_pretrained(model_name)
        self.device = device
        self.half = bool(fp16) and device.type == "cuda"
        self.model = model.to(device).eval()
        if self.half:
            self.model.half()
        for param in self.model.parameters():
            param.requires_grad_(False)
        self.model_name = model_name or type(model).__name__
        self.n_layers = self.model.config.num_hidden_layers + 1  # + the CNN feature-projection output
        self.hidden_size = self.model.config.hidden_size

    @property
    def feature_dim(self):
        return 2 * self.hidden_size  # mean and std pooling

    def _normalize(self, waves):
        x = waves.to(self.device, torch.float32)
        return (x - x.mean(dim=1, keepdim=True)) / torch.sqrt(x.var(dim=1, keepdim=True, unbiased=False) + 1e-7)

    @torch.no_grad()
    def pooled(self, waves):
        """[batch, samples] float waveforms -> [batch, layers, 2*hidden] pooled features.

        Inputs are normalized to zero mean / unit variance per clip, which is what the wav2vec 2.0 /
        XLS-R feature extractors do (`do_normalize=True`).
        """
        x = self._normalize(waves)
        out = self.model(x.half() if self.half else x, output_hidden_states=True)
        hidden = torch.stack(out.hidden_states, dim=1).float()  # [batch, layers, frames, hidden]
        return torch.cat([hidden.mean(dim=2), hidden.std(dim=2)], dim=-1)

    def mean_only(self, pooled_features):
        """[*, 2*hidden] concatenated mean+std pooling -> [*, hidden] mean-only half (Phase 2C Model D:
        an ablation of `pooled`'s std half, not a new encoder pass -- see `build_ssl_cache`'s cache)."""
        return pooled_features[..., :self.hidden_size]

    @torch.no_grad()
    def sequence(self, waves, layer):
        """[batch, samples] -> [batch, frames, hidden] per-frame hidden states of one layer index
        (0 = the CNN feature-projection output, same indexing as `hidden_states`). Phase 2C Model E
        needs the full frame sequence, not the pooled summary `pooled` gives; caching every layer's
        full sequence is infeasible (see the notebook's cost note), so only one layer is kept."""
        x = self._normalize(waves)
        out = self.model(x.half() if self.half else x, output_hidden_states=True)
        return out.hidden_states[layer].float()


def build_ssl_cache(encoder, waves_int16, cache_path, batch_size=16, progress_every=2000):
    """Pooled features for every clip in `waves_int16` -> float16 memmap [N, layers, 2*hidden].
    Reuses an existing file of the right shape (e.g. after a kernel restart)."""
    from numpy.lib.format import open_memmap

    cache_path = Path(cache_path)
    shape = (len(waves_int16), encoder.n_layers, encoder.feature_dim)
    done_marker = cache_path.with_suffix(".done")
    if cache_path.exists() and done_marker.exists():
        existing = np.load(cache_path, mmap_mode="r")
        if existing.shape == shape:
            print(f"reusing SSL feature cache {cache_path}")
            return existing
    out = open_memmap(cache_path, mode="w+", dtype=np.float16, shape=shape)
    for start in range(0, len(waves_int16), batch_size):
        batch = torch.from_numpy(np.asarray(waves_int16[start:start + batch_size], dtype=np.float32) / INT16_SCALE)
        out[start:start + len(batch)] = encoder.pooled(batch).cpu().numpy().astype(np.float16)
        done = start + len(batch)
        if progress_every and (done // batch_size) % max(1, progress_every // batch_size) == 0:
            print(f"  SSL features {done}/{len(waves_int16)}")
    out.flush()
    done_marker.write_text("ok", encoding="utf-8")
    return np.load(cache_path, mmap_mode="r")


def build_ssl_sequence_cache(encoder, waves_int16, layer, cache_path, batch_size=16, progress_every=2000):
    """Per-frame hidden states of one layer for every clip -> float16 memmap [N, frames, hidden]
    (Phase 2C Model E). Frame count is fixed because every input clip is the same fixed sample count.
    Reuses an existing file of the right shape, same as `build_ssl_cache`."""
    from numpy.lib.format import open_memmap

    cache_path = Path(cache_path)
    done_marker = cache_path.with_suffix(".done")
    with torch.no_grad():
        probe = encoder.sequence(torch.from_numpy(np.asarray(waves_int16[:1], dtype=np.float32) / INT16_SCALE), layer)
    n_frames = probe.shape[1]
    shape = (len(waves_int16), n_frames, encoder.hidden_size)
    if cache_path.exists() and done_marker.exists():
        existing = np.load(cache_path, mmap_mode="r")
        if existing.shape == shape:
            print(f"reusing SSL sequence cache {cache_path}")
            return existing
    out = open_memmap(cache_path, mode="w+", dtype=np.float16, shape=shape)
    for start in range(0, len(waves_int16), batch_size):
        batch = torch.from_numpy(np.asarray(waves_int16[start:start + batch_size], dtype=np.float32) / INT16_SCALE)
        out[start:start + len(batch)] = encoder.sequence(batch, layer).cpu().numpy().astype(np.float16)
        done = start + len(batch)
        if progress_every and (done // batch_size) % max(1, progress_every // batch_size) == 0:
            print(f"  SSL sequence features {done}/{len(waves_int16)}")
    out.flush()
    done_marker.write_text("ok", encoding="utf-8")
    return np.load(cache_path, mmap_mode="r")


def tiny_wav2vec2_for_smoke():
    """A randomly initialised, tiny wav2vec 2.0 -- exercises the real transformers code path in the
    local smoke test without downloading a checkpoint. Never used for reported numbers."""
    from transformers import Wav2Vec2Config, Wav2Vec2Model

    config = Wav2Vec2Config(hidden_size=32, num_hidden_layers=2, num_attention_heads=2, intermediate_size=64,
                            conv_dim=(16,) * 7, num_conv_pos_embeddings=16, num_conv_pos_embedding_groups=4)
    torch.manual_seed(0)
    return Wav2Vec2Model(config)

## Task 1 -- data: every bonafide clip, a capped spoof sample, and the attack IDs

In [ ]:
ITW_ARCHIVE = os.environ.get("TRUSTMESH_ITW_ARCHIVE", IN_THE_WILD_ARCHIVE)
ITW_PROTOCOL = os.environ.get("TRUSTMESH_ITW_PROTOCOL", "hf")

t0 = time.time()
if SMOKE_TEST:
    # Bonafide pads long enough that the real 0.2 s silence window fits; spoof pads stay far longer,
    # so the planted silence leakage is still there for the ablations to find.
    synthetic = make_synthetic_asvspoof(240, n_attacks=6, bonafide_pad_s=(0.12, 0.3))
    table = None
    meta = {"label": np.array([c["label"] for c in synthetic]),
            "utterance_id": [c["utterance_id"] for c in synthetic],
            "speaker_id": [c["speaker_id"] for c in synthetic],
            "columns": ["path", "audio", "label", "notes"], "note_keys": ["speaker_id", "utterance_id"]}
    protocol_path = write_synthetic_protocol(synthetic, OUTPUT_DIR / "synthetic_protocol.txt")
    dataset_name = "synthetic (smoke test)"
else:
    print(f"downloading {ASVSPOOF_DATASET} (4.39 GB parquet, cached by `datasets`) ...")
    table = load_asvspoof_table()
    meta = table_metadata(table)
    protocol_path = None
    dataset_name = f"{ASVSPOOF_DATASET} (split '{ASVSPOOF_SPLIT}' = LA evaluation partition)"
print(f"metadata for {len(meta['label'])} clips in {time.time() - t0:.0f}s")

# Log what the dataset itself exposes before reaching for the protocol file.
schema = {"columns": meta["columns"], "note_keys": meta["note_keys"],
          "attack_id_in_dataset": any(k in ("attack", "attack_id", "system_id") for k in meta["columns"] + meta["note_keys"])}
print("dataset schema:", schema)

try:
    protocol, protocol_source = load_eval_protocol(path=protocol_path)
    verification = verify_protocol(protocol, meta["utterance_id"], meta["label"], meta["speaker_id"])
except Exception as exc:  # noqa: BLE001 -- no protocol means no attack IDs, which is a recorded outcome
    protocol, protocol_source = {}, None
    verification = {"usable": False, "error": f"{type(exc).__name__}: {exc}"}
ATTACKS_AVAILABLE = bool(verification["usable"])
print("protocol verification:", verification)
print("attack-disjoint protocol:", "AVAILABLE" if ATTACKS_AVAILABLE else "NOT available -- unseen attacks stay unmeasured")

all_attacks = [protocol[u]["attack"] if u in protocol else "unknown" for u in meta["utterance_id"]]
selected = select_subset(meta["label"], SPOOF_PER_BONAFIDE, seed=SEED)
labels_np = meta["label"][selected].astype(np.int64)
speakers = [meta["speaker_id"][i] for i in selected]
attacks = [all_attacks[i] for i in selected]
N = len(selected)

from collections import Counter

data_summary = {
    "dataset": dataset_name,
    "schema": schema,
    "protocol_source": protocol_source,
    "protocol_verification": verification,
    "attacks_available": ATTACKS_AVAILABLE,
    "partition_counts": {"clips": int(len(meta["label"])), "bonafide": int((meta["label"] == BONAFIDE).sum()),
                         "spoof": int((meta["label"] == SPOOF).sum())},
    "spoof_per_bonafide_requested": SPOOF_PER_BONAFIDE,
    # The *actual* loaded counts -- the numbers to quote, never the requested ones.
    "loaded": {"clips": int(N), "bonafide": int((labels_np == BONAFIDE).sum()), "spoof": int((labels_np == SPOOF).sum()),
               "speakers": len(set(speakers))},
    "attack_counts_loaded": dict(sorted(Counter(attacks).items())),
    "phase2_loaded_for_comparison": {"clips": 6000, "bonafide": 619, "spoof": 5381},
}
print(json.dumps(data_summary["loaded"], indent=2))
save_progress("data", data_summary)

In [ ]:
# One decode pass -> on-disk cache of every input the rest of the notebook needs. After a crash, rerunning
# this cell reuses the cache instead of decoding ~37k clips again.
def iter_selected_clips():
    if SMOKE_TEST:
        for i in selected:
            yield synthetic[i]
    else:
        for row in table.select(selected.tolist()):
            wave, sr = _decode_audio(row["audio"])
            yield {"waveform": wave, "sample_rate": sr}


extractor = LFCCExtractor().to(DEVICE)
t0 = time.time()
cache = build_feature_cache(iter_selected_clips(), N, CACHE_DIR, extractor, DEVICE, CLIP_SAMPLES, SILENCE_SAMPLES,
                            fixed_silence_s=FIXED_SILENCE_S, noise_dbfs=NOISE_DBFS, seed=SEED,
                            config_key=f"{dataset_name}|{SEED}|{SPOOF_PER_BONAFIDE}|{N}|{int(selected.sum())}",
                            progress_every=None if SMOKE_TEST else 2000)
print(f"feature cache ready in {time.time() - t0:.0f}s, {cache_size_gb(CACHE_DIR):.2f} GB in {CACHE_DIR}")

silence_by_class = {}
for key in ("lead_silence_s", "trail_silence_s", "speech_s", "silence_ratio"):
    values = np.array([s[key] for s in cache["stats"]])
    silence_by_class[key] = {"bonafide_mean": float(values[labels_np == BONAFIDE].mean()),
                             "spoof_mean": float(values[labels_np == SPOOF].mean())}
probe_before = duration_leakage_probe(cache["stats"], labels_np)
save_progress("silence_stats", {"by_class": silence_by_class, "timing_probe_before_trim": probe_before,
                                "timing_probe_verdict": leakage_verdict(probe_before),
                                "clips_with_silence_window": int(cache["has_silence_window"].sum())})
print("silence by class:", json.dumps(silence_by_class, indent=1))

## Splits

**P1 -- speaker-disjoint, shared attacks** (the base protocol, as in Phase 2).
**P2 -- speaker- *and* attack-disjoint:** the *same* speaker partition, with spoofs from the held-out
attacks removed from train/val and the test set restricted to bonafide + held-out-attack spoofs.
Because the P2 test set is a subset of the P1 test set, a P1 model ("attacks seen in training") and a
P2 model ("attacks unseen") can be compared on identical clips.

In [ ]:
meta_clips = [{"speaker_id": speakers[i], "label": int(labels_np[i]), "attack": attacks[i]} for i in range(N)]


def sub_split(indices, fraction, seed):
    subset = [meta_clips[i] for i in indices]
    left, right = speaker_disjoint_split(subset, fraction, seed)
    return [indices[i] for i in left], [indices[i] for i in right]


def counts_of(idx):
    lab = labels_np[np.asarray(idx, dtype=np.int64)]
    return {"bonafide": int((lab == BONAFIDE).sum()), "spoof": int((lab == SPOOF).sum()), "total": int(len(lab))}


def has_both_classes(idx):
    c = counts_of(idx)
    return c["bonafide"] > 0 and c["spoof"] > 0


trainval, test = speaker_disjoint_split(meta_clips, TEST_FRACTION, SEED)
train, val = sub_split(trainval, VAL_FRACTION, SEED + 1)
SPLITS = {"P1": {"train": train, "val": val, "test": test}}
held_out, held_out_draws = [], 0
if ATTACKS_AVAILABLE:
    # Held-out attacks are a seeded draw. A draw is rejected only if it leaves some P2 split without a
    # class (e.g. every val spoof came from held-out attacks) -- a structural constraint, not tuning.
    for attempt in range(20):
        candidate = choose_held_out_attacks(attacks, N_HELD_OUT_ATTACKS, seed=SEED + attempt)
        p2 = {"train": drop_attacks(train, attacks, candidate), "val": drop_attacks(val, attacks, candidate),
              "test": keep_only_attacks(test, attacks, candidate)}
        if all(has_both_classes(p2[part]) for part in p2):
            held_out, held_out_draws, SPLITS["P2"] = candidate, attempt + 1, p2
            break
    if "P2" not in SPLITS:
        ATTACKS_AVAILABLE = False
        RESULTS["not_run"]["attack_disjoint_protocol"] = "no held-out-attack draw kept both classes in every P2 split"
else:
    RESULTS["not_run"]["attack_disjoint_protocol"] = ("attack IDs unavailable or failed verification -- "
                                                      "unseen-attack generalization remains unmeasured")

split_summary = {}
for name, split in SPLITS.items():
    train_speakers = {speakers[i] for i in split["train"]}
    test_speakers = {speakers[i] for i in split["test"]}
    split_summary[name] = {
        part: {**counts_of(split[part]), "attacks": sorted({attacks[i] for i in split[part]} - {"bonafide"})}
        for part in ("train", "val", "test")
    }
    split_summary[name]["shared_speakers_train_test"] = len(train_speakers & test_speakers)
    split_summary[name]["attack_overlap_train_test"] = sorted(
        set(split_summary[name]["train"]["attacks"]) & set(split_summary[name]["test"]["attacks"]))
    for part in ("train", "val", "test"):
        c = split_summary[name][part]
        print(f"{name} {part:5s}: {c['bonafide']:5d} bonafide / {c['spoof']:5d} spoof | attacks {c['attacks']}")
    print(f"{name} shared speakers train/test: {split_summary[name]['shared_speakers_train_test']} | "
          f"attack overlap train/test: {split_summary[name]['attack_overlap_train_test'] or 'none'}")
    for part in ("train", "val", "test"):
        c = split_summary[name][part]
        if not (c["bonafide"] and c["spoof"]):
            raise RuntimeError(f"{name} {part} split is missing a class: {c}")
if split_summary["P1"]["shared_speakers_train_test"]:
    print("WARNING: P1 fell back to a random split -- the headline is NOT speaker-disjoint")
save_progress("splits", {**split_summary, "held_out_attacks": held_out, "held_out_draws_needed": held_out_draws})

## Task 3 -- models, one variable per run (on P1)

A: LCNN, trimmed LFCC | B: A + augmentation (training batches only) | C: frozen XLS-R + layer-weighted head.

In [ ]:
MODEL_RESULTS, SCORES, MODELS = {}, {}, {}
MODEL_RESULTS_2C, SCORES_2C, MODELS_2C = {}, {}, {}  # Phase 2C's own store -- see `run()`'s `store` param
LCNN_FRAMES = extractor.n_frames(CLIP_SAMPLES)
SILENCE_FRAMES = extractor.n_frames(SILENCE_SAMPLES)
ssl_features, ssl_info = None, {}


def fetch_for(features, augment=False, seed=SEED):
    if features == "trimmed":
        return wave_lfcc_source(cache["wave_trimmed"], extractor, DEVICE, augment=augment, seed=seed)
    if features == "silence_only":
        return lfcc_array_source(cache["lfcc_silence_only"], DEVICE)
    if features == "ssl":
        return array_source(ssl_features, DEVICE)
    return lfcc_array_source(cache["lfcc"][features], DEVICE)


def build_model(kind, features):
    if kind == "ssl":
        return SSLHead(ssl_info["n_layers"], ssl_info["feature_dim"])
    return LCNN(n_features=extractor.n_features, n_frames=SILENCE_FRAMES if features == "silence_only" else LCNN_FRAMES)


def run(name, group, kind, features, split_name, seed, augment=False, role="", split=None,
        batch_size=None, model_override=None, fetch_override=None, store=None, epochs=None, verbose=False):
    """Train on train, select on val, then score the test split exactly once.

    `store` = (results_dict, models_dict, scores_dict, save_fn), defaulting to the Phase 2B globals so
    every existing call site is unchanged. Phase 2C passes its own store so its runs never touch
    results_phase2b.json (the spec says the two phases' output files must stay separable).

    `verbose=True` (Phase 2C's Model F, the one run here slow enough that Kaggle's log otherwise shows
    nothing for a long time between "training ..." and the result line) prints one line per epoch, so
    the log stays visibly alive instead of looking stuck.
    """
    results_dict, models_dict, scores_dict, save_fn = store or (MODEL_RESULTS, MODELS, SCORES, save_progress)
    split = split or SPLITS[split_name]
    print(f"training {name} ({role}) ...")
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = model_override if model_override is not None else build_model(kind, features)
    fetch = fetch_override if fetch_override is not None else fetch_for(features, augment, seed)
    lab = labels_np[np.asarray(split["train"])]
    weight = float((lab == SPOOF).sum() / max((lab == BONAFIDE).sum(), 1))
    bs = batch_size if batch_size is not None else BATCH_SIZE
    is_ssl_kind = kind in ("ssl", "ssl_seq")
    ep = epochs if epochs is not None else (SSL_EPOCHS if is_ssl_kind else EPOCHS)
    t_start = time.time()
    fit = train_model(model, fetch, labels_np, split["train"], weight, DEVICE, val_idx=split["val"],
                      epochs=ep, batch_size=bs,
                      lr=SSL_LR if is_ssl_kind else LEARNING_RATE,
                      patience=SSL_PATIENCE if is_ssl_kind else PATIENCE, seed=seed, verbose=verbose)
    test_scores = score_model(model, fetch, split["test"], batch_size=bs)
    test_labels = labels_np[np.asarray(split["test"])]
    ci = bootstrap_eer_ci(test_scores, test_labels, n_resamples=N_BOOTSTRAP, seed=seed)
    record = {
        "group": group, "kind": kind, "features": features, "split": split_name, "seed": seed,
        "augment": augment, "role": role, "test_eer": ci["eer"], "test_ci": [ci["ci_low"], ci["ci_high"]],
        "test_counts": counts_of(split["test"]), "train_counts": counts_of(split["train"]),
        "best_val_eer": fit["best_val_eer"], "best_epoch": fit["best_epoch"], "epochs_run": fit["epochs_run"],
        "validation_usable": fit["validation_usable"], "train_seconds": round(time.time() - t_start, 1),
        "n_params": model.n_params() if hasattr(model, "n_params") else None,
    }
    if kind == "ssl":
        record["layer_weights"] = model.layer_weights()
    results_dict[name], models_dict[name] = record, model
    scores_dict[name] = {"test_idx": np.asarray(split["test"]), "scores": test_scores, "labels": test_labels}
    save_fn("models", results_dict)
    print(f"{name:34s} val EER {fit['best_val_eer'] if fit['best_val_eer'] is not None else float('nan'):.4f} | "
          f"test EER {ci['eer']:.4f} [{ci['ci_low']:.4f}, {ci['ci_high']:.4f}] | "
          f"{record['test_counts']['bonafide']} bf / {record['test_counts']['spoof']} sp | "
          f"{fit['epochs_run']} ep {record['train_seconds']:.0f}s")
    return record


# Run A: data alone (the headline), 3 seeds.
for seed in SEEDS:
    run(f"A_lcnn_trimmed_P1_s{seed}", "A_lcnn_trimmed", "lcnn", "trimmed", "P1", seed, role="headline")

# Run B: A + augmentation, first seed (more seeds below if it wins on validation).
run(f"B_lcnn_trimmed_aug_P1_s{SEEDS[0]}", "B_lcnn_trimmed_aug", "lcnn", "trimmed", "P1", SEEDS[0], augment=True,
    role="augmentation")

In [ ]:
# Run C: frozen SSL front-end. Encoder runs once over every clip; pooled layer features are cached.
t0 = time.time()
ssl_encoder = None
try:
    if SMOKE_TEST:
        ssl_encoder = SSLEncoder(model=tiny_wav2vec2_for_smoke(), device=DEVICE, fp16=False)
        ssl_info["model"] = "tiny random wav2vec2 (smoke test)"
    else:
        for candidate in (SSL_MODEL, SSL_FALLBACK_MODEL):
            try:
                ssl_encoder = SSLEncoder(candidate, device=DEVICE, fp16=True)
                ssl_info["model"] = candidate
                break
            except Exception as exc:  # noqa: BLE001 -- try the fallback, record why
                ssl_info.setdefault("load_failures", []).append(f"{candidate}: {type(exc).__name__}: {exc}")
    if ssl_encoder is None:
        raise RuntimeError(f"no SSL encoder could be loaded: {ssl_info.get('load_failures')}")
    ssl_features = build_ssl_cache(ssl_encoder, cache["wave_trimmed"], CACHE_DIR / "ssl_pooled.npy",
                                   batch_size=SSL_BATCH, progress_every=None if SMOKE_TEST else 4000)
    ssl_info.update({"n_layers": ssl_encoder.n_layers, "feature_dim": ssl_encoder.feature_dim,
                     "pooling": "per-layer mean + std over frames", "frozen": True,
                     "extract_seconds": round(time.time() - t0, 1)})
    for seed in SEEDS:
        run(f"C_ssl_P1_s{seed}", "C_ssl_frozen", "ssl", "ssl", "P1", seed, role="frozen SSL front-end")
except Exception as exc:  # noqa: BLE001
    ssl_info["error"] = f"{type(exc).__name__}: {exc}"
    ssl_info["traceback"] = traceback.format_exc()
    RESULTS["not_run"]["run_C_ssl"] = ssl_info["error"]
    print("Run C failed, continuing without it:", ssl_info["error"])
save_progress("ssl", ssl_info)

In [ ]:
# Model selection on validation only: compare the first seed of each family.
first_seed_runs = {}
for run_name, record in MODEL_RESULTS.items():
    if record["seed"] == SEEDS[0] and record["split"] == "P1":
        first_seed_runs[record["group"]] = run_name
val_by_group = {g: MODEL_RESULTS[n]["best_val_eer"] for g, n in first_seed_runs.items()
                if MODEL_RESULTS[n]["best_val_eer"] is not None}
best_group = min(val_by_group, key=val_by_group.get)
print("validation EER by family (first seed):", {g: round(v, 4) for g, v in val_by_group.items()})
print("best family on validation:", best_group)

# The best family gets 3 seeds too (A and C already have them).
if best_group == "B_lcnn_trimmed_aug" and within_budget("extra_seeds_B"):
    for seed in SEEDS[1:]:
        run(f"B_lcnn_trimmed_aug_P1_s{seed}", "B_lcnn_trimmed_aug", "lcnn", "trimmed", "P1", seed, augment=True,
            role="augmentation")

# Untrimmed: the silence shortcut, kept only as a comparison. Never a result.
run(f"U_lcnn_untrimmed_P1_s{SEEDS[0]}", "U_lcnn_untrimmed_SHORTCUT", "lcnn", "untrimmed", "P1", SEEDS[0],
    role="shortcut comparison -- NOT a result")
save_progress("model_selection", {"validation_eer_first_seed": val_by_group, "best_group": best_group,
                                  "rule": "lowest validation EER (P1 val split), first seed of each family"})

## Task 2 -- attack-disjoint (P2): does it work on attacks it never saw?

In [ ]:
ATTACK_GAPS = {}
if ATTACKS_AVAILABLE and within_budget("attack_disjoint_runs"):
    family_runs = {"A_lcnn_trimmed": ("lcnn", "trimmed", False), "B_lcnn_trimmed_aug": ("lcnn", "trimmed", True)}
    if "C_ssl_frozen" in val_by_group:
        family_runs["C_ssl_frozen"] = ("ssl", "ssl", False)
    p1_test_pos = {clip: k for k, clip in enumerate(SPLITS["P1"]["test"])}
    p2_test = SPLITS["P2"]["test"]
    for group, (kind, features, augment) in family_runs.items():
        p2_name = f"{group}_P2_s{SEEDS[0]}"
        run(p2_name, f"{group}_P2", kind, features, "P2", SEEDS[0], augment=augment, role="attack-disjoint")
        seen_name = first_seed_runs[group]
        seen_scores = SCORES[seen_name]["scores"][[p1_test_pos[c] for c in p2_test]]  # slice, don't rescore
        unseen_scores = SCORES[p2_name]["scores"]
        p2_labels = labels_np[np.asarray(p2_test)]
        gap = bootstrap_eer_difference(unseen_scores, p2_labels, seen_scores, p2_labels, n_resamples=N_BOOTSTRAP,
                                       seed=SEED, paired=True)
        p2_attacks = [attacks[i] for i in p2_test]
        ATTACK_GAPS[group] = {
            "test_counts": counts_of(p2_test), "held_out_attacks": held_out,
            "eer_attacks_seen_in_training": compute_eer(seen_scores, p2_labels)["eer"],
            "eer_attacks_unseen": compute_eer(unseen_scores, p2_labels)["eer"],
            "unseen_minus_seen": gap,
            "per_attack_seen": per_attack_eer(seen_scores, p2_labels, p2_attacks),
            "per_attack_unseen": per_attack_eer(unseen_scores, p2_labels, p2_attacks),
        }
        g = ATTACK_GAPS[group]
        print(f"{group:20s} same {g['test_counts']['bonafide']} bf / {g['test_counts']['spoof']} sp test clips: "
              f"seen {g['eer_attacks_seen_in_training']:.4f} -> unseen {g['eer_attacks_unseen']:.4f} "
              f"(gap {gap['difference']:+.4f}, CI [{gap['ci_low']:+.4f}, {gap['ci_high']:+.4f}])")
    save_progress("attack_disjoint", ATTACK_GAPS)

## Task 4 -- silence: what is the shortcut made of?

All ablations use the same P1 split and are compared with paired bootstraps on the same test clips.

In [ ]:
SILENCE = {}
if within_budget("silence_ablations"):
    for mode in SILENCE_ABLATIONS:
        run(f"S_{mode}_P1_s{SEEDS[0]}", f"S_{mode}", "lcnn", mode, "P1", SEEDS[0], role="silence ablation")

    eligible = cache["has_silence_window"]
    silence_split = {part: [i for i in SPLITS["P1"][part] if eligible[i]] for part in ("train", "val", "test")}
    silence_ok = all(counts_of(silence_split[p])["bonafide"] and counts_of(silence_split[p])["spoof"]
                     for p in ("train", "val", "test"))
    if silence_ok:
        run(f"S_silence_only_P1_s{SEEDS[0]}", "S_silence_only", "lcnn", "silence_only", "P1", SEEDS[0],
            role="silence ablation", split=silence_split)
    else:
        RESULTS["not_run"]["silence_only"] = f"too few clips with {SILENCE_WINDOW_S}s of silence in a class: " + \
            json.dumps({p: counts_of(silence_split[p]) for p in silence_split})

    trimmed_ref, untrimmed_ref = first_seed_runs["A_lcnn_trimmed"], f"U_lcnn_untrimmed_P1_s{SEEDS[0]}"

    def paired(a, b, clips=None):
        """EER(a) - EER(b) on the same test clips (optionally restricted to `clips`)."""
        def take(name):
            idx, sc = SCORES[name]["test_idx"], SCORES[name]["scores"]
            if clips is None:
                return sc, labels_np[idx]
            pos = {c: k for k, c in enumerate(idx)}
            return sc[[pos[c] for c in clips]], labels_np[np.asarray(clips)]
        sa, la = take(a)
        sb, lb = take(b)
        return bootstrap_eer_difference(sa, la, sb, lb, n_resamples=N_BOOTSTRAP, seed=SEED, paired=True)

    comparisons = {"untrimmed_minus_trimmed": paired(untrimmed_ref, trimmed_ref)}
    for mode in SILENCE_ABLATIONS:
        name = f"S_{mode}_P1_s{SEEDS[0]}"
        comparisons[f"{mode}_minus_trimmed"] = paired(name, trimmed_ref)
        comparisons[f"{mode}_minus_untrimmed"] = paired(name, untrimmed_ref)
    silence_only_name = f"S_silence_only_P1_s{SEEDS[0]}"
    if silence_only_name in SCORES:
        test_e = silence_split["test"]
        so = SCORES[silence_only_name]
        comparisons["silence_only_ci"] = bootstrap_eer_ci(so["scores"], so["labels"], n_resamples=N_BOOTSTRAP, seed=SEED)
        # Same clips (those with enough silence) for every model in this comparison.
        comparisons["untrimmed_minus_trimmed_on_silence_subset"] = paired(untrimmed_ref, trimmed_ref, clips=test_e)
        comparisons["silence_only_minus_trimmed_on_silence_subset"] = paired(silence_only_name, trimmed_ref, clips=test_e)
        comparisons["silence_only_minus_untrimmed_on_silence_subset"] = paired(silence_only_name, untrimmed_ref, clips=test_e)

    # Verdicts, each stated with what it does NOT show.
    def significant(d, sign):
        return d["excludes_zero"] and (d["difference"] < 0 if sign < 0 else d["difference"] > 0)

    findings = []
    shortcut = comparisons["untrimmed_minus_trimmed"]
    shortcut_present = significant(shortcut, -1)
    findings.append(("shortcut", f"Untrimmed vs trimmed on the same clips: {shortcut['difference']:+.4f} EER "
                     f"(CI [{shortcut['ci_low']:+.4f}, {shortcut['ci_high']:+.4f}]) -- "
                     + ("the silence shortcut is present." if shortcut_present
                        else "no significant shortcut here, so there is nothing for the duration/replacement "
                             "ablations below to decompose; their numbers are recorded, not interpreted.")))
    dur = comparisons["zeros_original_duration_minus_trimmed"]
    dur_vs_untrimmed = comparisons["zeros_original_duration_minus_untrimmed"]
    if shortcut_present:
        if significant(dur, -1):
            text = (f"Silence replaced by digital zeros *at its original length* still beats trimmed by "
                    f"{-dur['difference']:.4f} EER, so silence duration/position alone reaches the model.")
            text += (f" It still trails untrimmed by {dur_vs_untrimmed['difference']:.4f} EER (CI excludes zero), so "
                     f"the silence's content adds information on top of its length." if significant(dur_vs_untrimmed, +1)
                     else " It is not significantly worse than untrimmed, so duration accounts for most of the shortcut.")
        else:
            text = ("Zeros at the original silence length do not beat trimmed significantly, so duration/position "
                    "alone does not reproduce the shortcut.")
        findings.append(("duration", text + " Does NOT show: which property of the content matters."))
    for mode in ("zeros_fixed", "noise_fixed"):
        d = comparisons[f"{mode}_minus_trimmed"]
        findings.append((mode, f"{mode}: {d['difference']:+.4f} EER vs trimmed (CI [{d['ci_low']:+.4f}, {d['ci_high']:+.4f}]) -- "
                         + ("not significantly different from trimmed" if not d["excludes_zero"]
                            else "significantly different from trimmed, so the replacement itself carries or disrupts signal")
                         + ". Does NOT separate duration from content on its own."))
    if "silence_only_ci" in comparisons:
        so_ci = comparisons["silence_only_ci"]
        carries = so_ci["ci_high"] < 0.5
        findings.append(("content", f"A classifier given only {SILENCE_WINDOW_S}s of each clip's genuine silence "
                         f"(fixed length, so no duration information) reaches EER {so_ci['eer']:.4f} "
                         f"[{so_ci['ci_low']:.4f}, {so_ci['ci_high']:.4f}] on {so_ci['n_bonafide']} bf / {so_ci['n_spoof']} sp "
                         + ("-- the silence *content* carries label information beyond duration."
                            if carries else "-- not distinguishable from chance, so silence content alone is not informative.")
                         + " Does NOT show: which acoustic property (noise floor, spectral shape, codec) it is, and it "
                         "only covers clips with at least that much silence."))
    SILENCE = {"comparisons": comparisons, "findings": [{"test": k, "text": t} for k, t in findings]}
    for _, t in findings:
        print("-", t)
    save_progress("silence_ablation", SILENCE)

## Free the LFCC variant caches Task 3/4 needed and nothing after this needs

`cache["lfcc"]` (untrimmed + the 3 silence ablations) and `cache["lfcc_silence_only"]` are only read
by the runs just above; every model score is already saved in `SCORES`, so the raw arrays aren't
needed again -- not by Task 5/6 below, and not by Phase 2C, which extends this notebook in the same
Kaggle session. Freeing them here (~7 GB) rather than only at the very end is what makes room for
Phase 2C's own cache needs later in this session (issue #25: an earlier run overran Kaggle's disk
because nothing was freed until the end).

In [ ]:
import gc

_freed_gb = 0.0
for _variant in list(cache["lfcc"].keys()):
    cache["lfcc"][_variant] = None
cache["lfcc_silence_only"] = None
gc.collect()
for _path in [CACHE_DIR / f"lfcc_{v}.npy" for v in LFCC_VARIANTS] + [CACHE_DIR / "lfcc_silence_only.npy"]:
    if _path.exists():
        _freed_gb += _path.stat().st_size / 1e9
        _path.unlink()
print(f"freed {_freed_gb:.2f} GB of LFCC variant caches no longer needed after Task 4")

## Task 5 -- rigour: seeds, CIs, and how much the CI shrank

In [ ]:
def group_summary(group):
    runs = [r for r in MODEL_RESULTS.values() if r["group"] == group]
    eers = [r["test_eer"] for r in runs]
    first = min(runs, key=lambda r: r["seed"])
    return {"group": group, "kind": first["kind"], "features": first["features"], "split": first["split"],
            "role": first["role"], "seeds": [r["seed"] for r in runs], "eer_per_seed": eers,
            "eer_mean": float(np.mean(eers)), "eer_std": float(np.std(eers)) if len(eers) > 1 else None,
            "ci_first_seed": first["test_ci"], "test_counts": first["test_counts"], "train_counts": first["train_counts"]}


LEADERBOARD = sorted((group_summary(g) for g in {r["group"] for r in MODEL_RESULTS.values()}),
                     key=lambda row: (row["split"], row["eer_mean"]))
headline = group_summary("A_lcnn_trimmed")
best = group_summary(best_group)
headline_first = first_seed_runs["A_lcnn_trimmed"]
ci_full = bootstrap_eer_ci(SCORES[headline_first]["scores"], SCORES[headline_first]["labels"], n_resamples=N_BOOTSTRAP, seed=SEED)
ci_phase2_size = bootstrap_ci_at_counts(SCORES[headline_first]["scores"], SCORES[headline_first]["labels"],
                                        PHASE2_TEST_COUNTS["bonafide"], PHASE2_TEST_COUNTS["spoof"],
                                        n_resamples=N_BOOTSTRAP, seed=SEED)
ci_shrinkage = {"ci_width_at_phase2b_test_size": ci_full["ci_width"], "phase2b_test_counts": headline["test_counts"],
                "ci_width_at_phase2_test_size": ci_phase2_size["ci_width"], "phase2_test_counts": PHASE2_TEST_COUNTS,
                "width_ratio": ci_full["ci_width"] / ci_phase2_size["ci_width"] if ci_phase2_size["ci_width"] else None,
                "method": "same headline model's test scores, bootstrapped at Phase 2's class counts vs the full Phase 2B test set"}
print(f"{'group':30s} {'split':5s} {'seeds':>5s} {'EER mean':>9s} {'std':>7s}  CI (first seed)       test bf/sp")
for row in LEADERBOARD:
    std = f"{row['eer_std']:.4f}" if row["eer_std"] is not None else "   -  "
    print(f"{row['group']:30s} {row['split']:5s} {len(row['seeds']):5d} {row['eer_mean']:9.4f} {std:>7s}  "
          f"[{row['ci_first_seed'][0]:.4f}, {row['ci_first_seed'][1]:.4f}]   "
          f"{row['test_counts']['bonafide']}/{row['test_counts']['spoof']}")
ratio_text = f"{ci_shrinkage['width_ratio']:.2f}" if ci_shrinkage["width_ratio"] is not None else "n/a"
print(f"\nCI width on the headline: {ci_full['ci_width']:.4f} at {headline['test_counts']['bonafide']} bonafide test clips vs "
      f"{ci_phase2_size['ci_width']:.4f} at Phase 2's {PHASE2_TEST_COUNTS['bonafide']} (ratio {ratio_text})")
save_progress("leaderboard", LEADERBOARD)
save_progress("ci_shrinkage", ci_shrinkage)

## Task 6 -- In-the-Wild, rerun with the fixed loader

In [ ]:
ITW = {"status": "not_run"}
if within_budget("in_the_wild"):
    try:
        archive = ZipAudioArchive(ITW_ARCHIVE, protocol=ITW_PROTOCOL)
        itw_clips, itw_report = load_archive_clips_balanced(archive, N_ITW_PER_CLASS, seed=SEED)
        itw_labels = np.array([c["label"] for c in itw_clips])
        trimmer = VoicePreprocessor(clip_seconds=CLIP_SECONDS, trim=True)
        raw_pre = VoicePreprocessor(clip_seconds=CLIP_SECONDS, trim=False)
        itw_waves = np.stack([to_int16(trimmer.prepare_waveform(c["waveform"], c["sample_rate"])[0]) for c in itw_clips])
        with torch.no_grad():
            itw_untrimmed = np.stack([
                extractor(torch.from_numpy(quantize(raw_pre.prepare_waveform(c["waveform"], c["sample_rate"])[0]))
                          .unsqueeze(0).to(DEVICE)).half().cpu().numpy()[0] for c in itw_clips])
        itw_fetch = {"trimmed": wave_lfcc_source(itw_waves, extractor, DEVICE),
                     "untrimmed": lfcc_array_source(itw_untrimmed, DEVICE)}
        if ssl_encoder is not None:
            itw_ssl = np.concatenate([ssl_encoder.pooled(torch.from_numpy(itw_waves[s:s + SSL_BATCH].astype(np.float32) / INT16_SCALE))
                                      .cpu().numpy().astype(np.float16) for s in range(0, len(itw_waves), SSL_BATCH)])
            itw_fetch["ssl"] = array_source(itw_ssl, DEVICE)

        to_score = {"headline_A": headline_first, "best_" + best_group: first_seed_runs[best_group],
                    "untrimmed_SHORTCUT": f"U_lcnn_untrimmed_P1_s{SEEDS[0]}"}
        if "C_ssl_frozen" in first_seed_runs:
            to_score["ssl_C"] = first_seed_runs["C_ssl_frozen"]
        itw_scores = {}
        for label, name in to_score.items():
            features = MODEL_RESULTS[name]["features"]
            sc = score_model(MODELS[name], itw_fetch[features], np.arange(len(itw_clips)))
            ci = bootstrap_eer_ci(sc, itw_labels, n_resamples=N_BOOTSTRAP, seed=SEED)
            itw_scores[label] = {"model": name, "eer": ci["eer"], "ci": [ci["ci_low"], ci["ci_high"]],
                                 "in_domain_eer": MODEL_RESULTS[name]["test_eer"]}
            print(f"In-the-Wild {label:24s} EER {ci['eer']:.4f} [{ci['ci_low']:.4f}, {ci['ci_high']:.4f}] "
                  f"(in-domain {MODEL_RESULTS[name]['test_eer']:.4f})")
        ITW = {"status": "ok", "source": ITW_ARCHIVE, "report": itw_report,
               "counts": {"bonafide": int((itw_labels == BONAFIDE).sum()), "spoof": int((itw_labels == SPOOF).sum())},
               "models": itw_scores}
    except Exception as exc:  # noqa: BLE001 -- recorded as a new issue, never claimed as generalization
        ITW = {"status": "failed", "error": f"{type(exc).__name__}: {exc}", "traceback": traceback.format_exc()}
        print("In-the-Wild failed -- log this traceback as a new issue:\n", ITW["traceback"])
save_progress("in_the_wild", ITW)

## Save: DET curves, weights, scores, closing paragraph

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 6))
for name, record in MODEL_RESULTS.items():
    if record["seed"] != SEEDS[0]:
        continue
    det = det_curve_points(SCORES[name]["scores"], SCORES[name]["labels"])
    style = "--" if record["split"] == "P2" else ":" if record["group"].startswith(("U_", "S_")) else "-"
    plt.plot(np.array(det["far"]) * 100, np.array(det["frr"]) * 100, style,
             label=f"{record['group']} [{record['split']}] EER {record['test_eer']:.3f}")
plt.plot([0.01, 100], [0.01, 100], "k-", linewidth=0.5, alpha=0.4)
plt.xscale("log")
plt.yscale("log")
plt.xlabel("False accept rate (%) -- spoof accepted as bonafide")
plt.ylabel("False reject rate (%) -- bonafide rejected")
plt.title("Phase 2B DET curves (solid: P1, dashed: attack-disjoint P2, dotted: shortcut/ablations)")
plt.legend(fontsize=6, loc="lower left")
plt.grid(True, which="both", linewidth=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "det_curves_phase2b.png", dpi=130)
plt.show()

for name, model in MODELS.items():
    torch.save(model.state_dict(), OUTPUT_DIR / f"phase2b_{name}.pt")
np.savez_compressed(OUTPUT_DIR / "phase2b_test_scores.npz",
                    **{f"{n}__{f}": SCORES[n][f] for n in SCORES for f in ("test_idx", "scores", "labels")})

# --- Closing paragraph, generated from this run's numbers -------------------------------------------
parts = [
    f"Headline (LCNN on trimmed LFCC, speaker-disjoint, shared attacks): EER {headline['eer_mean']:.4f} +/- "
    f"{headline['eer_std']:.4f} over {len(headline['seeds'])} seeds (95% CI of the first seed "
    f"[{headline['ci_first_seed'][0]:.4f}, {headline['ci_first_seed'][1]:.4f}]) on {headline['test_counts']['bonafide']} "
    f"bonafide / {headline['test_counts']['spoof']} spoof test clips, trained on {data_summary['loaded']['bonafide']} "
    f"bonafide clips (Phase 2: 619). At Phase 2's test size the same model's CI would be "
    f"{ci_shrinkage['ci_width_at_phase2_test_size']:.4f} wide; here it is {ci_shrinkage['ci_width_at_phase2b_test_size']:.4f}.",
]
if best_group != "A_lcnn_trimmed":
    parts.append(f"Best family on validation: {best_group}, test EER {best['eer_mean']:.4f}"
                 + (f" +/- {best['eer_std']:.4f} over {len(best['seeds'])} seeds" if best["eer_std"] is not None else "")
                 + f" (CI [{best['ci_first_seed'][0]:.4f}, {best['ci_first_seed'][1]:.4f}]).")
else:
    parts.append("Neither augmentation nor the frozen SSL front-end beat the plain LCNN on validation.")
if ATTACK_GAPS:
    gap_text = "; ".join(
        f"{g}: {v['eer_attacks_seen_in_training']:.4f} seen -> {v['eer_attacks_unseen']:.4f} unseen "
        f"(gap {v['unseen_minus_seen']['difference']:+.4f}, CI [{v['unseen_minus_seen']['ci_low']:+.4f}, "
        f"{v['unseen_minus_seen']['ci_high']:+.4f}])" for g, v in ATTACK_GAPS.items())
    parts.append(f"Attack-disjoint (held-out {', '.join(held_out)}, same test clips): {gap_text}.")
else:
    parts.append("Attack-disjoint: not measured -- " + RESULTS["not_run"].get(
        "attack_disjoint_protocol", RESULTS["not_run"].get("attack_disjoint_runs", "not run")) + ".")
if SILENCE:
    parts.append("Silence ablations: " + " ".join(f["text"] for f in SILENCE["findings"]))
else:
    parts.append("Silence ablations: not run (" + RESULTS["not_run"].get("silence_ablations", "unknown") + ").")
if ITW.get("status") == "ok":
    itw_text = "; ".join(f"{k} {v['eer']:.4f} [{v['ci'][0]:.4f}, {v['ci'][1]:.4f}] (in-domain {v['in_domain_eer']:.4f})"
                         for k, v in ITW["models"].items())
    parts.append(f"In-the-Wild ({ITW['counts']['bonafide']} bonafide / {ITW['counts']['spoof']} spoof clips, "
                 f"{ITW['report']['failed']} failed to load): {itw_text}.")
else:
    parts.append(f"In-the-Wild: {ITW.get('status')} ({ITW.get('error', RESULTS['not_run'].get('in_the_wild', ''))}) "
                 "-- no generalization claim.")
parts.append("Scope limits that remain: every clip is from the LA evaluation partition (not the official "
             "train/eval protocol), so no comparison with published ASVspoof numbers is valid; attack-disjoint uses "
             f"{len(held_out)} held-out attacks from one seeded draw; the SSL encoder is frozen, not fine-tuned.")
closing_paragraph = " ".join(parts)
print(closing_paragraph)
save_progress("closing_paragraph", closing_paragraph)
save_progress("run_hours", round(elapsed_hours(), 2))
print(f"\nPhase 2B results: {RESULTS_PATH} | elapsed {elapsed_hours():.2f} h")

# Phase 2C -- Voice Engine: SSL head upgrade (Tier 1 only)

Implements `docs/PHASE_2C_VOICE_SSL_DEPTH.md`. Extends this notebook in place (same file, same
Kaggle session, same `TIME_BUDGET_HOURS` clock) rather than a new notebook, reusing Phase 2B's
frozen SSL encoder, cached features, splits and In-the-Wild data computed above. Writes its own
`results_phase2c.json`, never merged into `results_phase2b.json`.

Model C (Phase 2B's best, frozen XLS-R + layer-weighted head) is the fixed baseline. This phase
asks one question: does a richer head on top of the same frozen features close more of its 0.10
In-the-Wild EER, without touching the frozen encoder or reusing any held-out test set twice.

## Task 0 -- pre-flight: what Model C already contains

Model C's pooling is already logged above as `ssl_info["pooling"]` = `"per-layer mean + std over
frames"` -- the spec's premise that Model C pools mean-only does **not** hold in this codebase
(`SSLEncoder.pooled` in `ssl.py` concatenates mean and std before `SSLHead` ever sees it). Task 1 is
reframed per that finding: Model D ablates the std half of Model C's *own* pooling (mean-only),
instead of adding std as a "new" variable Model C already has.

In [ ]:
PHASE2C = {"prerequisite": None}
if "C_ssl_frozen" not in first_seed_runs or ssl_encoder is None:
    PHASE2C["prerequisite"] = ("Model C (Run C) did not complete -- Phase 2C needs its frozen encoder, "
                               "cache, and learned layer weights, so nothing below can run.")
    RESULTS2C["not_run"]["phase2c"] = PHASE2C["prerequisite"]
    save_progress_2c("not_run", RESULTS2C["not_run"])
    print("SKIPPING Phase 2C:", PHASE2C["prerequisite"])
else:
    c_name = first_seed_runs["C_ssl_frozen"]
    c_model = MODELS[c_name]
    c_layer_weights = c_model.layer_weights()
    sorted_layers = sorted(range(len(c_layer_weights)), key=lambda i: -c_layer_weights[i])
    half = len(c_layer_weights) // 2
    preflight = {
        "model_c_pooling": ssl_info.get("pooling"),
        "spec_premise_mean_only_holds": ssl_info.get("pooling") == "mean over frames",
        "layer_weights_sorted": [{"layer": i, "weight": c_layer_weights[i]} for i in sorted_layers],
        "top_layer": sorted_layers[0],
        "early_vs_late": ("early/middle layers carry more weight" if sum(c_layer_weights[:half]) > 0.5
                          else "late layers carry more weight"),
        "model_c_n_params": c_model.n_params(),
    }
    print("Model C pooling (already computed by Run C, not re-derived):", preflight["model_c_pooling"])
    print("Model C's learned layer weights, sorted heaviest first:", json.dumps(preflight["layer_weights_sorted"], indent=1))
    print(preflight["early_vs_late"] + " -- tests the 'early layers carry more spoofing-relevant detail' hypothesis.")
    save_progress_2c("preflight", preflight)

## Task 1 -- Model D: mean-only ablation of Model C's own pooling

The one-variable comparison the pre-flight finding makes honest: Model C (mean+std) vs. Model D
(mean only), on the *same* cached features -- no new encoder pass, since `ssl.py`'s
`SSLEncoder.pooled` already concatenates `[mean, std]` per layer and Model D just reads the first
half of the existing cache (`array_mean_only_source`).

In [ ]:
if PHASE2C["prerequisite"] is None:
    def fetch_D(seed):
        return array_mean_only_source(ssl_features, ssl_encoder.hidden_size, DEVICE)

    def build_D():
        return SSLHead(ssl_info["n_layers"], ssl_encoder.hidden_size)

## Task 2 -- Model E: a lightweight temporal model on frozen SSL tokens

Caching every layer's full per-frame sequence is infeasible: at ~200 frames x 1024-dim x
`ssl_info["n_layers"]` layers x 2 bytes (float16) x the loaded clip count, that is on the order of
hundreds of GB -- far past what Kaggle's `/kaggle/working` can hold. Model E therefore keeps only the
single most-weighted layer from Model C's learned weights above (logged in the pre-flight step), so
there is no per-layer weighting mechanism here (unlike `SSLHead`, which mixes every layer) -- that
trade-off is the point of this deviation from the spec's literal wording.

**Disk, not just compute, is the constraint (issue #25):** even a *single* layer's full-dataset
sequence cache is ~15 GB, and a first attempt that pre-built it as a disk cache (like every other SSL
run here) failed a real Kaggle run with "tried to use more disk space than is available" once stacked
on top of the feature/SSL caches this notebook already holds. Model E therefore reads its sequence
on the fly per batch instead -- `wave_ssl_sequence_source`, the same frozen-encoder-per-batch pattern
Model F already uses for a different reason (augmentation) -- trading GPU time for zero added disk.
A single-layer BiLSTM (chosen over a TCN: fewer moving parts to implement correctly, and
`torch.nn.LSTM` is a tested, batched primitive) is kept deliberately small (hidden size 64) so the
comparison tests temporal modeling, not added capacity.

**Issue #26 (a second real Kaggle finding, on top of #25):** the first fix for #25 made Model E read
its sequence on the fly *every* call, including the validation-EER check `train_model` runs every
epoch and the one final test-scoring call. Model E has **no augmentation at all**, so its features
for a given clip never change between calls -- recomputing them from the raw waveform every epoch was
pure wasted GPU time on top of the already-expensive per-batch encoder forward, and is very likely
why a real run was still going after 8+ hours. The correct fix, now that Task 4's early cache cleanup
(above) frees real headroom: build the sequence **once**, as a disk cache, the way every other SSL
run here already works -- the one-time extraction pass costs about what Run C's full-dataset pooled
cache cost in Phase 2B (~2-5 minutes on the whole loaded dataset), and every epoch after that reads
it, exactly like Model D and Model C do. Freed right after Model E's P2 run below (not held to the
end), so it doesn't coexist with Model F's own disk-light but GPU-heavy training.

In [ ]:
E_AVAILABLE = RUN_MODEL_E and PHASE2C["prerequisite"] is None and within_budget_2c("model_E_sequence_cache")
if not RUN_MODEL_E and PHASE2C["prerequisite"] is None:
    RESULTS2C["not_run"]["model_E"] = ("skipped this run (RUN_MODEL_E=False) -- its full-dataset sequence "
                                       "cache overran this environment's disk quota on a real attempt "
                                       "(issue #28); set TRUSTMESH_RUN_MODEL_E=1 to include it")
    save_progress_2c("not_run", RESULTS2C["not_run"])
    print("SKIPPING Model E:", RESULTS2C["not_run"]["model_E"])
if E_AVAILABLE:
    E_LAYER = preflight["top_layer"]
    t0 = time.time()
    try:
        e_seq_cache = build_ssl_sequence_cache(ssl_encoder, cache["wave_trimmed"], E_LAYER,
                                               CACHE_DIR / f"ssl_seq_layer{E_LAYER}.npy", batch_size=SSL_BATCH,
                                               progress_every=None if SMOKE_TEST else 4000)
        cache_cost = {"layer_used": E_LAYER, "shape": list(e_seq_cache.shape),
                      "single_layer_cache_gb": round(e_seq_cache.nbytes / 1e9, 2),
                      "extract_seconds": round(time.time() - t0, 1)}
        print("Model E sequence cache:", json.dumps(cache_cost, indent=1))
        save_progress_2c("model_e_cache_cost", cache_cost)

        def fetch_E(seed):
            return array_source(e_seq_cache, DEVICE)

        def build_E():
            return SSLSeqHead(ssl_encoder.hidden_size, lstm_hidden=64)
    except Exception as exc:  # noqa: BLE001 -- Model E is optional; record why, continue without it
        RESULTS2C["not_run"]["model_E"] = f"{type(exc).__name__}: {exc}"
        save_progress_2c("not_run", RESULTS2C["not_run"])
        print("Model E sequence cache failed, continuing without it:", RESULTS2C["not_run"]["model_E"])
        E_AVAILABLE = False

## Task 3 -- Model F: training-time augmentation on Model C's recipe

Same architecture as Model C (mean+std pooling, same layer-weighted head), but augmentation must
happen *before* the frozen encoder's forward pass -- unlike every other SSL run here, which reads
the pre-computed cache, Model F's **training** batches re-run the frozen encoder (`wave_ssl_source`).
Its validation/test calls do not: they are never augmented, so they read Phase 2B's already-built
`ssl_features` cache instead of recomputing identical features from the waveform every epoch (the
other half of issue #26 -- the same redundant-eval mistake Model E made). Augmentation is Run B's
menu (noise, gain, SpecAugment) plus one light codec/compression simulation that Phase 2B explicitly
skipped for time -- see `simulate_low_bitrate` in `preprocess.py` for why it is a documented mu-law +
bandwidth-reduction proxy, not a real MP3 round-trip.

**Still the slowest model here even after the #26 fix**, because every *training* batch genuinely
needs a live encoder forward pass -- that part can't be cached. Its epoch cap is lowered to
`SSL_EPOCHS_ONTHEFLY` (Model C converged in 13-44 epochs on the cached path, so this doesn't cut
training short in practice) and it prints one line per epoch (`verbose=True`) so a long run shows
visible progress in Kaggle's log instead of looking stuck between the start and result lines.

In [ ]:
if PHASE2C["prerequisite"] is None:
    def fetch_F(seed):
        return wave_ssl_source(cache["wave_trimmed"], ssl_encoder, DEVICE, augment=True, seed=seed,
                               codec_sim=True, pooled_cache=ssl_features)

    def build_F():
        return SSLHead(ssl_info["n_layers"], ssl_info["feature_dim"])

## Train D, E, F: P1 (3 seeds), P2 attack-disjoint (first seed), In-the-Wild

Same protocol Model C got in Phase 2B, so every comparison below is apples-to-apples.

In [ ]:
PHASE2C_FAMILIES = {}
if PHASE2C["prerequisite"] is None:
    STORE_2C = (MODEL_RESULTS_2C, MODELS_2C, SCORES_2C, save_progress_2c)
    # (group, role, build_fn, fetch_fn, augment, batch_size, epochs, verbose) -- epochs/verbose default
    # to run()'s own SSL_EPOCHS/False for D and E (cheap, cache-backed); Model F re-runs the frozen
    # encoder every batch, so it gets a lower epoch cap and per-epoch progress prints (see SSL_EPOCHS_ONTHEFLY).
    specs = [("D_ssl_mean_only", "mean-only pooling ablation", build_D, fetch_D, False, BATCH_SIZE, None, False)]
    if E_AVAILABLE:
        specs.append(("E_ssl_bilstm", "temporal model (BiLSTM) on frozen SSL tokens", build_E, fetch_E, False,
                      BATCH_SIZE, None, False))
    if RUN_MODEL_F:
        specs.append(("F_ssl_augmented", "training-time augmentation on Model C's recipe", build_F, fetch_F, True,
                      SSL_BATCH, SSL_EPOCHS_ONTHEFLY, True))
    else:
        RESULTS2C["not_run"]["model_F"] = ("skipped this run to keep total Kaggle time down (RUN_MODEL_F=False) "
                                           "-- set TRUSTMESH_RUN_MODEL_F=1 to include it; Model C/D/E are unaffected")
        save_progress_2c("not_run", RESULTS2C["not_run"])
        print("SKIPPING Model F:", RESULTS2C["not_run"]["model_F"])

    for group, role, build_fn, fetch_fn, augment, bs, ep, verb in specs:
        kind = "ssl_seq" if group == "E_ssl_bilstm" else "ssl"
        seed_names = []
        for seed in SEEDS:
            if not within_budget_2c(f"{group}_P1"):
                break
            name = f"{group}_P1_s{seed}"
            run(name, group, kind, group.lower(), "P1", seed, augment=augment, role=role,
               model_override=build_fn(), fetch_override=fetch_fn(seed), batch_size=bs, store=STORE_2C,
               epochs=ep, verbose=verb)
            seed_names.append(name)
        if seed_names:
            PHASE2C_FAMILIES[group] = {"kind": kind, "seed_names": seed_names, "first_seed": seed_names[0]}

## Attack-disjoint (P2): same test clips as Model C's comparison

In [ ]:
PHASE2C_ATTACK_GAPS = {}
if PHASE2C["prerequisite"] is None and ATTACKS_AVAILABLE and PHASE2C_FAMILIES and within_budget_2c("phase2c_attack_disjoint"):
    p1_test_pos = {clip: k for k, clip in enumerate(SPLITS["P1"]["test"])}
    p2_test = SPLITS["P2"]["test"]
    p2_labels = labels_np[np.asarray(p2_test)]
    for group, fam in PHASE2C_FAMILIES.items():
        spec = next(s for s in specs if s[0] == group)
        _, role, build_fn, fetch_fn, augment, bs, ep, verb = spec
        seed = SEEDS[0]
        p2_name = f"{group}_P2_s{seed}"
        run(p2_name, f"{group}_P2", fam["kind"], group.lower(), "P2", seed, augment=augment,
           role="attack-disjoint", model_override=build_fn(), fetch_override=fetch_fn(seed), batch_size=bs,
           store=STORE_2C, epochs=ep, verbose=verb)
        seen_scores = SCORES_2C[fam["first_seed"]]["scores"][[p1_test_pos[c] for c in p2_test]]
        unseen_scores = SCORES_2C[p2_name]["scores"]
        gap = bootstrap_eer_difference(unseen_scores, p2_labels, seen_scores, p2_labels, n_resamples=N_BOOTSTRAP,
                                       seed=SEED, paired=True)
        PHASE2C_ATTACK_GAPS[group] = {
            "test_counts": counts_of(p2_test), "held_out_attacks": held_out,
            "eer_attacks_seen_in_training": compute_eer(seen_scores, p2_labels)["eer"],
            "eer_attacks_unseen": compute_eer(unseen_scores, p2_labels)["eer"],
            "unseen_minus_seen": gap,
        }
        g = PHASE2C_ATTACK_GAPS[group]
        print(f"{group:20s} same {g['test_counts']['bonafide']} bf / {g['test_counts']['spoof']} sp test clips: "
              f"seen {g['eer_attacks_seen_in_training']:.4f} -> unseen {g['eer_attacks_unseen']:.4f} "
              f"(gap {gap['difference']:+.4f}, CI [{gap['ci_low']:+.4f}, {gap['ci_high']:+.4f}])")
    save_progress_2c("attack_disjoint", PHASE2C_ATTACK_GAPS)

## Free Model E's sequence cache -- nothing after this point reads it

In-the-Wild scores Model E by re-running the encoder directly on the ~200 ITW clips (cheap), not by
reading this cache, and Model F's remaining training never touches it either.

In [ ]:
if "e_seq_cache" in dir() and e_seq_cache is not None:
    e_seq_cache = None
    gc.collect()
    p = CACHE_DIR / f"ssl_seq_layer{E_LAYER}.npy"
    if p.exists():
        freed = p.stat().st_size / 1e9
        p.unlink()
        print(f"freed {freed:.2f} GB (Model E's sequence cache, no longer needed)")

## In-the-Wild: D, E, F scored on the same clips Model C was scored on

In [ ]:
PHASE2C_ITW = {}
if (PHASE2C["prerequisite"] is None and PHASE2C_FAMILIES and ITW.get("status") == "ok"
        and within_budget_2c("phase2c_in_the_wild")):
    # Recomputed (cheap: ~200 clips), not read back from ITW["models"], which only kept the EER/CI
    # summary, not the raw per-clip scores a paired comparison needs.
    c_itw_raw = score_model(MODELS[c_name], itw_fetch["ssl"], np.arange(len(itw_clips))) if "ssl" in itw_fetch else None
    try:
        itw_int16 = itw_waves  # trimmed, int16 -- same clips as Model C's ITW pass above
        for group, fam in PHASE2C_FAMILIES.items():
            model = MODELS_2C[fam["first_seed"]]
            if group == "D_ssl_mean_only":
                feats = ssl_encoder.mean_only(torch.from_numpy(np.asarray(itw_ssl, dtype=np.float32)))
                sc = score_model(model, lambda idx, training, feats=feats: feats[np.asarray(idx)].to(DEVICE),
                                 np.arange(len(itw_int16)))
            elif group == "E_ssl_bilstm":
                seq_chunks = [ssl_encoder.sequence(torch.from_numpy(itw_int16[s:s + SSL_BATCH].astype(np.float32) / INT16_SCALE), E_LAYER)
                             for s in range(0, len(itw_int16), SSL_BATCH)]
                itw_seq = torch.cat(seq_chunks).cpu()
                sc = score_model(model, lambda idx, training, itw_seq=itw_seq: itw_seq[np.asarray(idx)].to(DEVICE),
                                 np.arange(len(itw_int16)))
            else:  # F_ssl_augmented -- on-the-fly, no augmentation at eval (fetch_override has augment=True
                   # only applied when training=True, and score_model always calls with training=False)
                fetch = wave_ssl_source(itw_int16, ssl_encoder, DEVICE)
                sc = score_model(model, fetch, np.arange(len(itw_int16)))
            ci = bootstrap_eer_ci(sc, itw_labels, n_resamples=N_BOOTSTRAP, seed=SEED)
            in_domain = MODEL_RESULTS_2C[fam["first_seed"]]["test_eer"]
            entry = {"model": fam["first_seed"], "eer": ci["eer"], "ci": [ci["ci_low"], ci["ci_high"]],
                    "in_domain_eer": in_domain}
            if c_itw_raw is not None:
                vs_c = bootstrap_eer_difference(sc, itw_labels, c_itw_raw, itw_labels,
                                               n_resamples=N_BOOTSTRAP, seed=SEED, paired=True)
                entry["vs_model_c"] = {"difference": vs_c["difference"], "ci_low": vs_c["ci_low"],
                                       "ci_high": vs_c["ci_high"], "excludes_zero": vs_c["excludes_zero"]}
            PHASE2C_ITW[group] = entry
            print(f"In-the-Wild {group:20s} EER {ci['eer']:.4f} [{ci['ci_low']:.4f}, {ci['ci_high']:.4f}] "
                  f"(in-domain {in_domain:.4f})")
        save_progress_2c("in_the_wild", PHASE2C_ITW)
    except Exception as exc:  # noqa: BLE001 -- recorded, never claimed as generalization
        RESULTS2C["not_run"]["in_the_wild"] = f"{type(exc).__name__}: {exc}"
        save_progress_2c("not_run", RESULTS2C["not_run"])
        print("Phase 2C In-the-Wild failed:", RESULTS2C["not_run"]["in_the_wild"])
elif PHASE2C["prerequisite"] is None:
    RESULTS2C["not_run"]["in_the_wild"] = "Phase 2B's own In-the-Wild pass did not succeed, so there are no clips to reuse"
    save_progress_2c("not_run", RESULTS2C["not_run"])

## Results: one leaderboard, C carried over unchanged, plus D, E, F

In [ ]:
if PHASE2C["prerequisite"] is None:
    def group_summary_2c(group):
        runs = [r for r in MODEL_RESULTS_2C.values() if r["group"] == group]
        eers = [r["test_eer"] for r in runs]
        first = min(runs, key=lambda r: r["seed"])
        return {"group": group, "kind": first["kind"], "features": first["features"], "split": first["split"],
                "role": first["role"], "seeds": [r["seed"] for r in runs], "eer_per_seed": eers,
                "eer_mean": float(np.mean(eers)), "eer_std": float(np.std(eers)) if len(eers) > 1 else None,
                "ci_first_seed": first["test_ci"], "test_counts": first["test_counts"],
                "train_counts": first["train_counts"], "n_params": first["n_params"]}

    c_summary = group_summary("C_ssl_frozen")
    c_summary["n_params"] = c_model.n_params()
    PHASE2C_LEADERBOARD = [c_summary] + [group_summary_2c(g) for g in PHASE2C_FAMILIES]
    print(f"{'group':22s} {'seeds':>5s} {'EER mean':>9s} {'std':>7s}  CI (first seed)       params")
    for row in PHASE2C_LEADERBOARD:
        std = f"{row['eer_std']:.4f}" if row["eer_std"] is not None else "   -  "
        print(f"{row['group']:22s} {len(row['seeds']):5d} {row['eer_mean']:9.4f} {std:>7s}  "
              f"[{row['ci_first_seed'][0]:.4f}, {row['ci_first_seed'][1]:.4f}]   {row['n_params']}")
    save_progress_2c("leaderboard", PHASE2C_LEADERBOARD)

    # --- Closing paragraph ---------------------------------------------------------------------------
    c_itw_eer = ITW["models"][[k for k, v in ITW["models"].items() if v["model"] == c_name][0]]["eer"] \
        if ITW.get("status") == "ok" and any(v["model"] == c_name for v in ITW["models"].values()) else None
    parts_2c = [f"Model C carried over from Phase 2B: test EER {c_summary['eer_mean']:.4f}"
               + (f" +/- {c_summary['eer_std']:.4f}" if c_summary["eer_std"] is not None else "") + "."]
    improved = []
    for group in PHASE2C_FAMILIES:
        entry = PHASE2C_ITW.get(group)
        if entry and "vs_model_c" in entry and entry["vs_model_c"]["excludes_zero"] and entry["vs_model_c"]["difference"] < 0:
            improved.append(group)
    if c_itw_eer is None:
        parts_2c.append("In-the-Wild comparison unavailable: not run.")
    elif improved:
        parts_2c.append(f"Beat Model C's In-the-Wild EER {c_itw_eer:.4f} outside noise: {', '.join(improved)}.")
    else:
        parts_2c.append(f"None of {', '.join(PHASE2C_FAMILIES)} beat Model C's In-the-Wild EER {c_itw_eer:.4f} "
                        "outside noise -- Model C remains a legitimate final answer.")
    regressed = [g for g, v in PHASE2C_ATTACK_GAPS.items()
                if v["unseen_minus_seen"]["excludes_zero"] and v["unseen_minus_seen"]["difference"] > 0
                and v["unseen_minus_seen"]["difference"] > ATTACK_GAPS.get("C_ssl_frozen", {}).get(
                    "unseen_minus_seen", {}).get("difference", 0)]
    parts_2c.append(f"Attack-disjoint gap regressed vs Model C in: {', '.join(regressed)}." if regressed
                    else "No variant made the attack-disjoint gap significantly worse than Model C's.")
    param_text = "; ".join(f"{g}: {PHASE2C_FAMILIES[g] and MODEL_RESULTS_2C[PHASE2C_FAMILIES[g]['first_seed']]['n_params']} params"
                           for g in PHASE2C_FAMILIES)
    parts_2c.append(f"Added parameters vs Model C ({c_model.n_params()}): {param_text}.")
    closing_2c = " ".join(parts_2c)
    print(closing_2c)
    save_progress_2c("closing_paragraph", closing_2c)
    save_progress_2c("run_hours", round(elapsed_hours(), 2))

    plt.figure(figsize=(7, 6))
    for name, record in {**{c_name: MODEL_RESULTS[c_name]}, **MODEL_RESULTS_2C}.items():
        scores_src = SCORES if name == c_name else SCORES_2C
        if record["seed"] != SEEDS[0]:
            continue
        det = det_curve_points(scores_src[name]["scores"], scores_src[name]["labels"])
        style = "--" if record["split"] == "P2" else "-"
        plt.plot(np.array(det["far"]) * 100, np.array(det["frr"]) * 100, style,
                 label=f"{record['group']} [{record['split']}] EER {record['test_eer']:.3f}")
    plt.plot([0.01, 100], [0.01, 100], "k-", linewidth=0.5, alpha=0.4)
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("False accept rate (%) -- spoof accepted as bonafide")
    plt.ylabel("False reject rate (%) -- bonafide rejected")
    plt.title("Phase 2C DET curves: C, D, E, F (solid: P1, dashed: attack-disjoint P2)")
    plt.legend(fontsize=6, loc="lower left")
    plt.grid(True, which="both", linewidth=0.3)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "det_curves_phase2c.png", dpi=130)
    plt.show()

    for name, model in MODELS_2C.items():
        torch.save(model.state_dict(), OUTPUT_DIR / f"phase2c_{name}.pt")
    if SCORES_2C:
        np.savez_compressed(OUTPUT_DIR / "phase2c_test_scores.npz",
                            **{f"{n}__{f}": SCORES_2C[n][f] for n in SCORES_2C for f in ("test_idx", "scores", "labels")})
    print(f"\nPhase 2C results: {RESULTS2C_PATH} | elapsed {elapsed_hours():.2f} h")

## Delete the feature cache (after both phases are done with it)

In [ ]:
if DELETE_CACHE_AT_END and CACHE_DIR.exists():
    import gc

    size = cache_size_gb(CACHE_DIR)
    cache = ssl_features = None  # release the memory-mapped files first; open memmaps can't be deleted on every OS
    gc.collect()
    shutil.rmtree(CACHE_DIR, ignore_errors=True)
    if CACHE_DIR.exists():
        print(f"WARNING: could not fully delete {CACHE_DIR} ({cache_size_gb(CACHE_DIR):.2f} GB left) -- "
              "delete it before committing if output size matters")
    else:
        print(f"deleted feature cache ({size:.2f} GB) so the committed output stays small")
print(f"\nresults: {RESULTS_PATH} (Phase 2B) and {RESULTS2C_PATH} (Phase 2C) | total elapsed {elapsed_hours():.2f} h")